In [1]:
# ======================================================================
# ECOBOT-X KNOWLEDGE DISTILLATION
# SECTION 1 — CONFIGURATION + TEACHER VERIFICATION
# ======================================================================

from pathlib import Path
import random
import numpy as np
import torch
import yaml

from ultralytics import YOLO


# ======================================================================
# 1. DATASET PATHS
# ======================================================================

DATASET_YAML = Path(
    r"G:\EcoBotX_YOLO\dataset.yaml"
)

TRAIN_IMAGES = Path(
    r"G:\EcoBotX_YOLO\images\train"
)

VAL_IMAGES = Path(
    r"G:\EcoBotX_YOLO\images\val"
)

TEST_IMAGES = Path(
    r"G:\EcoBotX_YOLO\images\test"
)

TRAIN_LABELS = Path(
    r"G:\EcoBotX_YOLO\labels\train"
)

VAL_LABELS = Path(
    r"G:\EcoBotX_YOLO\labels\val"
)

TEST_LABELS = Path(
    r"G:\EcoBotX_YOLO\labels\test"
)


# ======================================================================
# 2. TEACHER MODEL
# ======================================================================

TEACHER_PATH = Path(
    r"G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt"
)


# ======================================================================
# 3. OUTPUT DIRECTORY
# ======================================================================

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training"
    r"\knowledge_distillation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ======================================================================
# 4. STUDENT CONFIGURATION
# ======================================================================

STUDENT_NAME = "EcoBotX_Tiny_KD"

NUM_CLASSES = 4

CLASS_NAMES = [
    "BOTTLE",
    "CAN",
    "PAPER",
    "WRAPPER"
]


# ======================================================================
# 5. TRAINING CONFIGURATION
# ======================================================================

IMAGE_SIZE = 640

BATCH_SIZE = 8

EPOCHS = 100

LEARNING_RATE = 0.001

WEIGHT_DECAY = 0.0005

NUM_WORKERS = 4

SEED = 42


# ======================================================================
# 6. KNOWLEDGE DISTILLATION CONFIGURATION
# ======================================================================

KD_TEMPERATURE = 4.0

KD_FEATURE_WEIGHT = 1.0


# ======================================================================
# 7. DEVICE
# ======================================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ======================================================================
# 8. REPRODUCIBILITY
# ======================================================================

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)


# ======================================================================
# 9. PRINT ENVIRONMENT
# ======================================================================

print("=" * 70)
print("ECOBOT-X KNOWLEDGE DISTILLATION")
print("SECTION 1 — CONFIGURATION + TEACHER VERIFICATION")
print("=" * 70)

print()

print("PyTorch version :", torch.__version__)

print("CUDA available  :", torch.cuda.is_available())

print("Device          :", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU             :",
        torch.cuda.get_device_name(0)
    )

    gpu_memory = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / (1024 ** 3)
    )

    print(
        "GPU memory      :",
        f"{gpu_memory:.2f} GB"
    )

print()


# ======================================================================
# 10. PATH VALIDATION
# ======================================================================

print("=" * 70)
print("PATH VALIDATION")
print("=" * 70)

paths_to_check = {

    "Dataset YAML": DATASET_YAML,

    "Train images": TRAIN_IMAGES,

    "Validation images": VAL_IMAGES,

    "Test images": TEST_IMAGES,

    "Train labels": TRAIN_LABELS,

    "Validation labels": VAL_LABELS,

    "Test labels": TEST_LABELS,

    "Teacher best.pt": TEACHER_PATH
}


all_paths_valid = True


for name, path in paths_to_check.items():

    if path.exists():

        print(
            f"[OK] {name}"
        )

        print(
            f"     {path}"
        )

    else:

        print(
            f"[ERROR] {name} NOT FOUND"
        )

        print(
            f"        {path}"
        )

        all_paths_valid = False


print()


if not all_paths_valid:

    raise FileNotFoundError(
        "\nOne or more required paths are missing."
    )


# ======================================================================
# 11. DATASET YAML VERIFICATION
# ======================================================================

print("=" * 70)
print("DATASET YAML VERIFICATION")
print("=" * 70)

with open(
    DATASET_YAML,
    "r",
    encoding="utf-8"
) as f:

    dataset_config = yaml.safe_load(f)


yaml_names = dataset_config.get(
    "names",
    []
)


print()

print("Dataset classes:")


if isinstance(yaml_names, dict):

    yaml_names = {
        int(k): v
        for k, v in yaml_names.items()
    }

    for idx in sorted(yaml_names):

        print(
            f"  {idx}: {yaml_names[idx]}"
        )

else:

    for idx, name in enumerate(yaml_names):

        print(
            f"  {idx}: {name}"
        )


print()


if len(yaml_names) != NUM_CLASSES:

    raise ValueError(
        f"Dataset has {len(yaml_names)} classes, "
        f"but NUM_CLASSES={NUM_CLASSES}."
    )


print(
    f"[OK] Dataset contains {NUM_CLASSES} classes."
)


# ======================================================================
# 12. TEACHER CHECKPOINT
# ======================================================================

teacher_size_mb = (
    TEACHER_PATH.stat().st_size
    / (1024 ** 2)
)


print()

print("=" * 70)
print("TEACHER CHECKPOINT")
print("=" * 70)

print()

print(
    "Teacher architecture : YOLOv8n"
)

print(
    "Teacher path         :",
    TEACHER_PATH
)

print(
    "Checkpoint size      :",
    f"{teacher_size_mb:.2f} MB"
)

print(
    "[OK] Teacher checkpoint exists."
)


# ======================================================================
# 13. FINAL CONFIGURATION
# ======================================================================

print()

print("=" * 70)
print("FINAL KD CONFIGURATION")
print("=" * 70)

print()

print("TEACHER")
print("-" * 70)

print(
    "YOLOv8n:",
    TEACHER_PATH
)

print()

print("STUDENT")
print("-" * 70)

print(
    "Student name:",
    STUDENT_NAME
)

print()

print("DATASET")
print("-" * 70)

print(
    "Classes:",
    NUM_CLASSES
)

print(
    "Class names:",
    CLASS_NAMES
)

print()

print("TRAINING")
print("-" * 70)

print(
    "Image size:",
    IMAGE_SIZE
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Epochs:",
    EPOCHS
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "Workers:",
    NUM_WORKERS
)

print(
    "Device:",
    DEVICE
)

print()

print("KNOWLEDGE DISTILLATION")
print("-" * 70)

print(
    "Temperature:",
    KD_TEMPERATURE
)

print(
    "Feature KD weight:",
    KD_FEATURE_WEIGHT
)

print()

print("OUTPUT")
print("-" * 70)

print(
    OUTPUT_DIR
)

print()

print("=" * 70)
print("[OK] SECTION 1 COMPLETED")
print("=" * 70)

print()

print(
    "Ready for SECTION 2 — "
    "EcoBotX lightweight student architecture."
)

ECOBOT-X KNOWLEDGE DISTILLATION
SECTION 1 — CONFIGURATION + TEACHER VERIFICATION

PyTorch version : 2.11.0+cu128
CUDA available  : True
Device          : cuda
GPU             : NVIDIA GeForce RTX 3050 Laptop GPU
GPU memory      : 4.00 GB

PATH VALIDATION
[OK] Dataset YAML
     G:\EcoBotX_YOLO\dataset.yaml
[OK] Train images
     G:\EcoBotX_YOLO\images\train
[OK] Validation images
     G:\EcoBotX_YOLO\images\val
[OK] Test images
     G:\EcoBotX_YOLO\images\test
[OK] Train labels
     G:\EcoBotX_YOLO\labels\train
[OK] Validation labels
     G:\EcoBotX_YOLO\labels\val
[OK] Test labels
     G:\EcoBotX_YOLO\labels\test
[OK] Teacher best.pt
     G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt

DATASET YAML VERIFICATION

Dataset classes:
  0: BOTTLE
  1: CAN
  2: PAPER
  3: WRAPPER

[OK] Dataset contains 4 classes.

TEACHER CHECKPOINT

Teacher architecture : YOLOv8n
Teacher path         : G:\EcoBotX_YOLO_training\yolov8n_ecobotx\weights\best.pt
Checkpoint size      : 5.97 MB
[OK] Teac

In [2]:
# ======================================================================
# ECOBOT-X KNOWLEDGE DISTILLATION
# SECTION 2 — LIGHTWEIGHT STUDENT ARCHITECTURE
# ======================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


print("=" * 70)
print("ECOBOT-X KNOWLEDGE DISTILLATION")
print("SECTION 2 — LIGHTWEIGHT STUDENT ARCHITECTURE")
print("=" * 70)

print()
print("Device:", DEVICE)


# ======================================================================
# 1. GHOST CONV
# ======================================================================

class GhostConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=1,
        stride=1,
        activation=True
    ):

        super().__init__()

        padding = kernel_size // 2

        primary_channels = (
            out_channels + 1
        ) // 2

        cheap_channels = (
            out_channels -
            primary_channels
        )


        self.primary = nn.Sequential(

            nn.Conv2d(
                in_channels,
                primary_channels,
                kernel_size,
                stride,
                padding,
                bias=False
            ),

            nn.BatchNorm2d(
                primary_channels
            ),

            nn.ReLU6(
                inplace=True
            )
        )


        self.cheap = None


        if cheap_channels > 0:

            # ----------------------------------------------------------
            # IMPORTANT FIX
            # ----------------------------------------------------------
            # The original implementation can produce an invalid
            # groups configuration when cheap_channels is smaller
            # than primary_channels.
            #
            # We use groups=1 in that case.
            # ----------------------------------------------------------

            groups = (
                primary_channels
                if cheap_channels % primary_channels == 0
                else 1
            )


            self.cheap = nn.Sequential(

                nn.Conv2d(
                    primary_channels,
                    cheap_channels,
                    kernel_size=3,
                    stride=1,
                    padding=1,
                    groups=groups,
                    bias=False
                ),

                nn.BatchNorm2d(
                    cheap_channels
                ),

                nn.ReLU6(
                    inplace=True
                )
            )


        self.out_channels = out_channels


    def forward(self, x):

        y = self.primary(x)


        if self.cheap is None:

            return y


        z = self.cheap(y)


        return torch.cat(
            [y, z],
            dim=1
        )


# ======================================================================
# 2. DEPTHWISE SEPARABLE CONVOLUTION
# ======================================================================

class DWConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):

        super().__init__()


        self.depthwise = nn.Sequential(

            nn.Conv2d(
                in_channels,
                in_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=in_channels,
                bias=False
            ),

            nn.BatchNorm2d(
                in_channels
            ),

            nn.ReLU6(
                inplace=True
            )
        )


        self.pointwise = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU6(
                inplace=True
            )
        )


    def forward(self, x):

        x = self.depthwise(x)

        x = self.pointwise(x)

        return x


# ======================================================================
# 3. COORDINATE ATTENTION
# ======================================================================

class CoordinateAttention(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16
    ):

        super().__init__()


        reduced_channels = max(
            8,
            channels // reduction
        )


        self.conv1 = nn.Conv2d(
            channels,
            reduced_channels,
            kernel_size=1,
            bias=False
        )


        self.bn1 = nn.BatchNorm2d(
            reduced_channels
        )


        self.act = nn.ReLU6(
            inplace=True
        )


        self.conv_h = nn.Conv2d(
            reduced_channels,
            channels,
            kernel_size=1
        )


        self.conv_w = nn.Conv2d(
            reduced_channels,
            channels,
            kernel_size=1
        )


    def forward(self, x):

        identity = x

        b, c, h, w = x.shape


        # Height pooling

        x_h = x.mean(
            dim=3,
            keepdim=True
        )


        # Width pooling

        x_w = x.mean(
            dim=2,
            keepdim=True
        )


        x_w = x_w.permute(
            0,
            1,
            3,
            2
        )


        y = torch.cat(
            [x_h, x_w],
            dim=2
        )


        y = self.conv1(y)

        y = self.bn1(y)

        y = self.act(y)


        y_h, y_w = torch.split(
            y,
            [h, w],
            dim=2
        )


        y_w = y_w.permute(
            0,
            1,
            3,
            2
        )


        attention_h = torch.sigmoid(
            self.conv_h(y_h)
        )


        attention_w = torch.sigmoid(
            self.conv_w(y_w)
        )


        return (
            identity
            * attention_h
            * attention_w
        )


# ======================================================================
# 4. ECOBLOCK
# ======================================================================

class EcoBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1,
        attention=False
    ):

        super().__init__()


        self.ghost = GhostConv(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=1
        )


        self.dwconv = DWConv(
            out_channels,
            out_channels,
            stride=stride
        )


        if attention:

            self.attention = (
                CoordinateAttention(
                    out_channels
                )
            )

        else:

            self.attention = nn.Identity()


        if (
            stride != 1
            or
            in_channels != out_channels
        ):

            self.shortcut = nn.Sequential(

                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),

                nn.BatchNorm2d(
                    out_channels
                )
            )

        else:

            self.shortcut = nn.Identity()


        self.act = nn.ReLU6(
            inplace=True
        )


    def forward(self, x):

        identity = self.shortcut(x)


        y = self.ghost(x)

        y = self.dwconv(y)

        y = self.attention(y)

        y = y + identity

        y = self.act(y)


        return y


# ======================================================================
# 5. ECOBOT-X BACKBONE
# ======================================================================

class EcoBotXBackbone(nn.Module):

    def __init__(self):

        super().__init__()


        # --------------------------------------------------------------
        # Stem
        # 640 -> 320
        # --------------------------------------------------------------

        self.stem = GhostConv(
            3,
            32,
            kernel_size=3,
            stride=2
        )


        # --------------------------------------------------------------
        # Stage 1
        # 320 -> 160
        # --------------------------------------------------------------

        self.stage1 = EcoBlock(
            32,
            48,
            stride=2,
            attention=False
        )


        # --------------------------------------------------------------
        # Stage 2
        # 160 -> 80
        # P3
        # --------------------------------------------------------------

        self.stage2 = EcoBlock(
            48,
            64,
            stride=2,
            attention=False
        )


        self.stage2_refine = EcoBlock(
            64,
            64,
            stride=1,
            attention=False
        )


        # --------------------------------------------------------------
        # Stage 3
        # 80 -> 40
        # P4
        # --------------------------------------------------------------

        self.stage3 = EcoBlock(
            64,
            96,
            stride=2,
            attention=True
        )


        self.stage3_refine = EcoBlock(
            96,
            96,
            stride=1,
            attention=True
        )


        # --------------------------------------------------------------
        # Stage 4
        # 40 -> 20
        # P5
        # --------------------------------------------------------------

        self.stage4 = EcoBlock(
            96,
            128,
            stride=2,
            attention=True
        )


        self.stage4_refine = EcoBlock(
            128,
            128,
            stride=1,
            attention=True
        )


    def forward(self, x):

        x = self.stem(x)

        x = self.stage1(x)


        # P3

        x = self.stage2(x)

        p3 = self.stage2_refine(x)


        # P4

        x = self.stage3(p3)

        p4 = self.stage3_refine(x)


        # P5

        x = self.stage4(p4)

        p5 = self.stage4_refine(x)


        return p3, p4, p5


# ======================================================================
# 6. GHOST-PAN
# ======================================================================

class GhostPAN(nn.Module):

    def __init__(self):

        super().__init__()


        # --------------------------------------------------------------
        # TOP-DOWN P5 -> P4
        # --------------------------------------------------------------

        self.p5_to_p4 = GhostConv(
            128,
            96,
            kernel_size=1,
            stride=1
        )


        self.p4_fuse = GhostConv(
            192,
            96,
            kernel_size=1,
            stride=1
        )


        # --------------------------------------------------------------
        # TOP-DOWN P4 -> P3
        # --------------------------------------------------------------

        self.p4_to_p3 = GhostConv(
            96,
            64,
            kernel_size=1,
            stride=1
        )


        self.p3_fuse = GhostConv(
            128,
            64,
            kernel_size=1,
            stride=1
        )


        # --------------------------------------------------------------
        # BOTTOM-UP P3 -> P4
        # --------------------------------------------------------------

        self.p3_down = DWConv(
            64,
            96,
            stride=2
        )


        self.p4_pan = GhostConv(
            192,
            96,
            kernel_size=1,
            stride=1
        )


        # --------------------------------------------------------------
        # BOTTOM-UP P4 -> P5
        # --------------------------------------------------------------

        self.p4_down = DWConv(
            96,
            128,
            stride=2
        )


        self.p5_pan = GhostConv(
            256,
            128,
            kernel_size=1,
            stride=1
        )


    def forward(
        self,
        p3,
        p4,
        p5
    ):


        # ==============================================================
        # TOP-DOWN
        # ==============================================================

        p5_td = self.p5_to_p4(p5)


        p5_td = F.interpolate(
            p5_td,
            size=p4.shape[-2:],
            mode="nearest"
        )


        p4_td = torch.cat(
            [p4, p5_td],
            dim=1
        )


        p4_td = self.p4_fuse(
            p4_td
        )


        p4_up = self.p4_to_p3(
            p4_td
        )


        p4_up = F.interpolate(
            p4_up,
            size=p3.shape[-2:],
            mode="nearest"
        )


        p3_td = torch.cat(
            [p3, p4_up],
            dim=1
        )


        p3_out = self.p3_fuse(
            p3_td
        )


        # ==============================================================
        # BOTTOM-UP
        # ==============================================================

        p3_down = self.p3_down(
            p3_out
        )


        p4_out = torch.cat(
            [p3_down, p4_td],
            dim=1
        )


        p4_out = self.p4_pan(
            p4_out
        )


        p4_down = self.p4_down(
            p4_out
        )


        p5_out = torch.cat(
            [p4_down, p5],
            dim=1
        )


        p5_out = self.p5_pan(
            p5_out
        )


        return (
            p3_out,
            p4_out,
            p5_out
        )


# ======================================================================
# 7. ECOBOT-X STUDENT
# ======================================================================

class EcoBotXStudent(nn.Module):

    def __init__(
        self,
        num_classes=NUM_CLASSES
    ):

        super().__init__()


        self.num_classes = (
            num_classes
        )


        self.backbone = (
            EcoBotXBackbone()
        )


        self.neck = GhostPAN()


    def forward(self, x):

        # Backbone

        p3, p4, p5 = (
            self.backbone(x)
        )


        # Neck

        p3, p4, p5 = (
            self.neck(
                p3,
                p4,
                p5
            )
        )


        return {

            "P3": p3,

            "P4": p4,

            "P5": p5,

            "features": [
                p3,
                p4,
                p5
            ]
        }


# ======================================================================
# 8. CREATE STUDENT
# ======================================================================

print()
print("=" * 70)
print("CREATING ECOBOT-X STUDENT")
print("=" * 70)


student = EcoBotXStudent(
    num_classes=NUM_CLASSES
).to(DEVICE)


# ======================================================================
# 9. PARAMETER COUNT
# ======================================================================

total_parameters = sum(
    p.numel()
    for p in student.parameters()
)


trainable_parameters = sum(
    p.numel()
    for p in student.parameters()
    if p.requires_grad
)


parameter_size_mb = (
    total_parameters * 4
) / (
    1024 ** 2
)


print()

print(
    "Student parameters :",
    f"{total_parameters:,}"
)

print(
    "Trainable params   :",
    f"{trainable_parameters:,}"
)

print(
    "FP32 size          :",
    f"{parameter_size_mb:.3f} MB"
)


# ======================================================================
# 10. FORWARD PASS TEST
# ======================================================================

print()
print("=" * 70)
print("STUDENT FORWARD PASS TEST")
print("=" * 70)


student.eval()


dummy_input = torch.randn(
    1,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
    device=DEVICE
)


with torch.no_grad():

    output = student(
        dummy_input
    )


for name in [
    "P3",
    "P4",
    "P5"
]:

    print(
        f"{name}:",
        tuple(
            output[name].shape
        )
    )


# ======================================================================
# 11. STRICT ARCHITECTURE VALIDATION
# ======================================================================

expected_shapes = {

    "P3": (
        1,
        64,
        80,
        80
    ),

    "P4": (
        1,
        96,
        40,
        40
    ),

    "P5": (
        1,
        128,
        20,
        20
    )
}


print()
print("=" * 70)
print("ARCHITECTURE VALIDATION")
print("=" * 70)


architecture_valid = True


for name, expected in (
    expected_shapes.items()
):

    actual = tuple(
        output[name].shape
    )


    if actual == expected:

        print(
            f"[OK] {name}: {actual}"
        )

    else:

        print(
            f"[ERROR] {name}: "
            f"expected {expected}, "
            f"got {actual}"
        )

        architecture_valid = False


if not architecture_valid:

    raise RuntimeError(
        "Student architecture validation failed."
    )


# ======================================================================
# 12. STUDENT INFORMATION
# ======================================================================

print()
print("=" * 70)
print("STUDENT MODEL INFORMATION")
print("=" * 70)

print()

print(
    "Architecture       :",
    STUDENT_NAME
)

print(
    "Backbone           :",
    "Ghost + DWConv + Coordinate Attention"
)

print(
    "Neck               :",
    "Lightweight Ghost-PAN"
)

print(
    "Detection levels   :",
    "P3 / P4 / P5"
)

print(
    "P3 channels        :",
    64
)

print(
    "P4 channels        :",
    96
)

print(
    "P5 channels        :",
    128
)

print(
    "Input size         :",
    f"{IMAGE_SIZE}x{IMAGE_SIZE}"
)

print(
    "Total parameters   :",
    f"{total_parameters:,}"
)

print(
    "Parameters (M)     :",
    f"{total_parameters / 1e6:.3f}"
)

print(
    "FP32 size          :",
    f"{parameter_size_mb:.3f} MB"
)

print()

print(
    "[OK] STUDENT ARCHITECTURE "
    "VALIDATED SUCCESSFULLY."
)

print()

print(
    "P3 = [1, 64, 80, 80]"
)

print(
    "P4 = [1, 96, 40, 40]"
)

print(
    "P5 = [1, 128, 20, 20]"
)

print()

print("=" * 70)
print("[OK] SECTION 2 COMPLETED")
print("=" * 70)

print()

print(
    "Ready for SECTION 3 — "
    "Teacher / Student feature alignment."
)

ECOBOT-X KNOWLEDGE DISTILLATION
SECTION 2 — LIGHTWEIGHT STUDENT ARCHITECTURE

Device: cuda

CREATING ECOBOT-X STUDENT

Student parameters : 208,600
Trainable params   : 208,600
FP32 size          : 0.796 MB

STUDENT FORWARD PASS TEST
P3: (1, 64, 80, 80)
P4: (1, 96, 40, 40)
P5: (1, 128, 20, 20)

ARCHITECTURE VALIDATION
[OK] P3: (1, 64, 80, 80)
[OK] P4: (1, 96, 40, 40)
[OK] P5: (1, 128, 20, 20)

STUDENT MODEL INFORMATION

Architecture       : EcoBotX_Tiny_KD
Backbone           : Ghost + DWConv + Coordinate Attention
Neck               : Lightweight Ghost-PAN
Detection levels   : P3 / P4 / P5
P3 channels        : 64
P4 channels        : 96
P5 channels        : 128
Input size         : 640x640
Total parameters   : 208,600
Parameters (M)     : 0.209
FP32 size          : 0.796 MB

[OK] STUDENT ARCHITECTURE VALIDATED SUCCESSFULLY.

P3 = [1, 64, 80, 80]
P4 = [1, 96, 40, 40]
P5 = [1, 128, 20, 20]

[OK] SECTION 2 COMPLETED

Ready for SECTION 3 — Teacher / Student feature alignment.


In [3]:
# ======================================================================
# ECOBOT-X KNOWLEDGE DISTILLATION
# SECTION 3 — TEACHER / STUDENT FEATURE ALIGNMENT
# ======================================================================

import json
import torch
import torch.nn as nn
import torch.nn.functional as F


print("=" * 70)
print("ECOBOT-X KNOWLEDGE DISTILLATION")
print("SECTION 3 — TEACHER / STUDENT FEATURE ALIGNMENT")
print("=" * 70)


# ======================================================================
# 1. STUDENT CHANNELS
# ======================================================================

STUDENT_CHANNELS = {

    "P3": 64,

    "P4": 96,

    "P5": 128
}


EXPECTED_RESOLUTIONS = {

    "P3": (80, 80),

    "P4": (40, 40),

    "P5": (20, 20)
}


# ======================================================================
# 2. LOAD TEACHER
# ======================================================================

print()
print("=" * 70)
print("LOADING YOLOv8n TEACHER")
print("=" * 70)


teacher_wrapper = YOLO(
    str(TEACHER_PATH)
)


teacher = teacher_wrapper.model


teacher = teacher.to(
    DEVICE
)


teacher.eval()


# Freeze teacher

for parameter in teacher.parameters():

    parameter.requires_grad = False


print()

print(
    "[OK] Teacher loaded."
)

print(
    "[OK] Teacher frozen."
)

print(
    "Teacher type:",
    type(teacher).__name__
)


# ======================================================================
# 3. FEATURE COLLECTION
# ======================================================================

feature_records = []


def teacher_feature_hook(
    layer_index
):

    def hook(
        module,
        inputs,
        output
    ):

        tensors = []


        if torch.is_tensor(output):

            tensors.append(
                output
            )


        elif isinstance(
            output,
            (list, tuple)
        ):

            for item in output:

                if torch.is_tensor(item):

                    tensors.append(
                        item
                    )


        for tensor in tensors:

            if tensor.ndim == 4:

                feature_records.append({

                    "layer_index":
                        layer_index,

                    "layer_type":
                        module.__class__.__name__,

                    "tensor":
                        tensor

                })


    return hook


# ======================================================================
# 4. REGISTER TEACHER HOOKS
# ======================================================================

teacher_hooks = []


for index, layer in enumerate(
    teacher.model
):

    teacher_hooks.append(

        layer.register_forward_hook(

            teacher_feature_hook(
                index
            )
        )
    )


# ======================================================================
# 5. TEST TEACHER
# ======================================================================

dummy_input = torch.randn(
    1,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
    device=DEVICE
)


feature_records.clear()


with torch.no_grad():

    _ = teacher(
        dummy_input
    )


# ======================================================================
# 6. REMOVE HOOKS
# ======================================================================

for hook in teacher_hooks:

    hook.remove()


teacher_hooks.clear()


# ======================================================================
# 7. DISPLAY TEACHER FEATURE MAPS
# ======================================================================

print()
print("=" * 70)
print("TEACHER FEATURE MAP CANDIDATES")
print("=" * 70)


for i, record in enumerate(
    feature_records
):

    tensor = record["tensor"]


    print(

        f"[{i:03d}] "

        f"Layer={record['layer_index']:02d} | "

        f"{record['layer_type']:20s} | "

        f"Shape={tuple(tensor.shape)}"

    )


# ======================================================================
# 8. FIND FEATURE BY RESOLUTION
# ======================================================================

def find_feature_by_resolution(
    records,
    height,
    width
):

    candidates = []


    for record in records:

        tensor = record["tensor"]


        if (

            tensor.shape[2] == height

            and

            tensor.shape[3] == width

        ):

            candidates.append(
                record
            )


    if len(candidates) == 0:

        return None


    # Last suitable feature

    return candidates[-1]


# ======================================================================
# 9. SELECT P3 / P4 / P5
# ======================================================================

print()
print("=" * 70)
print("SEARCHING FOR TEACHER P3 / P4 / P5")
print("=" * 70)


teacher_features = {}


for level, resolution in (
    EXPECTED_RESOLUTIONS.items()
):

    h, w = resolution


    record = find_feature_by_resolution(

        feature_records,

        h,

        w
    )


    if record is None:

        raise RuntimeError(

            f"\nCould not find teacher "
            f"feature for {level} "
            f"with resolution "
            f"{h}x{w}."

        )


    teacher_features[level] = (
        record
    )


    tensor = record["tensor"]


    print(

        f"[OK] {level}: "

        f"Layer={record['layer_index']} | "

        f"{record['layer_type']} | "

        f"Shape={tuple(tensor.shape)}"

    )


# ======================================================================
# 10. TEACHER CHANNELS
# ======================================================================

TEACHER_CHANNELS = {}


for level in [
    "P3",
    "P4",
    "P5"
]:

    TEACHER_CHANNELS[level] = int(

        teacher_features[level]
        ["tensor"]
        .shape[1]

    )


print()
print("=" * 70)
print("TEACHER / STUDENT CHANNELS")
print("=" * 70)


for level in [
    "P3",
    "P4",
    "P5"
]:

    print(

        f"{level}: "

        f"Teacher={TEACHER_CHANNELS[level]} "

        f"-> "

        f"Student={STUDENT_CHANNELS[level]}"

    )


# ======================================================================
# 11. FEATURE ALIGNMENT
# ======================================================================

class FeatureAlign(nn.Module):

    def __init__(
        self,
        teacher_channels,
        student_channels
    ):

        super().__init__()


        self.projection = nn.Sequential(

            nn.Conv2d(

                teacher_channels,

                student_channels,

                kernel_size=1,

                stride=1,

                padding=0,

                bias=False
            ),

            nn.BatchNorm2d(
                student_channels
            )
        )


    def forward(self, x):

        return self.projection(x)


# ======================================================================
# 12. CREATE ALIGNMENT MODULES
# ======================================================================

alignment = nn.ModuleDict({

    "P3": FeatureAlign(

        TEACHER_CHANNELS["P3"],

        STUDENT_CHANNELS["P3"]

    ),

    "P4": FeatureAlign(

        TEACHER_CHANNELS["P4"],

        STUDENT_CHANNELS["P4"]

    ),

    "P5": FeatureAlign(

        TEACHER_CHANNELS["P5"],

        STUDENT_CHANNELS["P5"]

    )

}).to(DEVICE)


# ======================================================================
# 13. TEST ALIGNMENT
# ======================================================================

student.eval()

alignment.eval()


with torch.no_grad():

    student_output = student(
        dummy_input
    )


aligned_features = {}


for level in [
    "P3",
    "P4",
    "P5"
]:

    teacher_tensor = (
        teacher_features[level]
        ["tensor"]
    )


    aligned = alignment[level](
        teacher_tensor
    )


    aligned_features[level] = (
        aligned
    )


# ======================================================================
# 14. KD LOSS
# ======================================================================

def feature_kd_loss(
    student_feature,
    teacher_feature
):

    student_normalized = F.normalize(

        student_feature,

        p=2,

        dim=1
    )


    teacher_normalized = F.normalize(

        teacher_feature,

        p=2,

        dim=1
    )


    return F.mse_loss(

        student_normalized,

        teacher_normalized
    )


# ======================================================================
# 15. TEST KD LOSS
# ======================================================================

individual_losses = {}


total_kd_loss = torch.tensor(
    0.0,
    device=DEVICE
)


for level in [
    "P3",
    "P4",
    "P5"
]:

    loss = feature_kd_loss(

        student_output[level],

        aligned_features[level]
    )


    individual_losses[level] = (
        float(loss.detach().cpu())
    )


    total_kd_loss = (
        total_kd_loss
        + loss
    )


total_kd_loss = (
    total_kd_loss / 3.0
)


# ======================================================================
# 16. ALIGNMENT PARAMETER COUNT
# ======================================================================

alignment_parameters = sum(

    p.numel()

    for p in alignment.parameters()

)


alignment_size_mb = (

    alignment_parameters * 4

) / (

    1024 ** 2

)


# ======================================================================
# 17. VALIDATION
# ======================================================================

print()
print("=" * 70)
print("FEATURE ALIGNMENT VALIDATION")
print("=" * 70)


for level in [
    "P3",
    "P4",
    "P5"
]:

    teacher_shape = tuple(

        teacher_features[level]
        ["tensor"]
        .shape
    )


    aligned_shape = tuple(

        aligned_features[level]
        .shape
    )


    student_shape = tuple(

        student_output[level]
        .shape
    )


    print()

    print(
        f"{level}:"
    )

    print(
        "  Teacher :",
        teacher_shape
    )

    print(
        "  Aligned :",
        aligned_shape
    )

    print(
        "  Student :",
        student_shape
    )


# ======================================================================
# 18. SAVE SECTION 3 INFORMATION
# ======================================================================

section3_output = {

    "teacher": {

        "architecture":
            "YOLOv8n",

        "path":
            str(TEACHER_PATH),

        "features": {

            level: {

                "layer_index":
                    int(
                        teacher_features[level]
                        ["layer_index"]
                    ),

                "layer_type":
                    teacher_features[level]
                    ["layer_type"],

                "channels":
                    int(
                        teacher_features[level]
                        ["tensor"]
                        .shape[1]
                    ),

                "height":
                    int(
                        teacher_features[level]
                        ["tensor"]
                        .shape[2]
                    ),

                "width":
                    int(
                        teacher_features[level]
                        ["tensor"]
                        .shape[3]
                    )

            }

            for level in [
                "P3",
                "P4",
                "P5"
            ]
        }
    },


    "student": {

        "architecture":
            STUDENT_NAME,

        "features":
            STUDENT_CHANNELS

    },


    "alignment": {

        "P3":
            f"{TEACHER_CHANNELS['P3']} -> "
            f"{STUDENT_CHANNELS['P3']}",

        "P4":
            f"{TEACHER_CHANNELS['P4']} -> "
            f"{STUDENT_CHANNELS['P4']}",

        "P5":
            f"{TEACHER_CHANNELS['P5']} -> "
            f"{STUDENT_CHANNELS['P5']}"

    },


    "alignment_parameters":
        alignment_parameters,


    "alignment_size_mb":
        alignment_size_mb,


    "test_kd_loss":
        float(
            total_kd_loss
            .detach()
            .cpu()
        ),


    "individual_kd_loss":
        individual_losses,


    "status":
        "PASSED"
}


SECTION3_JSON = (

    OUTPUT_DIR /

    "section3_feature_alignment.json"

)


with open(
    SECTION3_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        section3_output,
        f,
        indent=2
    )


# ======================================================================
# 19. FINAL SUMMARY
# ======================================================================

print()
print("=" * 70)
print("SECTION 3 — FINAL SUMMARY")
print("=" * 70)

print()

for level in [
    "P3",
    "P4",
    "P5"
]:

    print(

        f"{level}: "

        f"Teacher "

        f"{tuple(teacher_features[level]['tensor'].shape)} "

        f"-> "

        f"Aligned "

        f"{tuple(aligned_features[level].shape)}"

    )


print()

print(
    "Teacher architecture : YOLOv8n"
)

print(
    "Student architecture :",
    STUDENT_NAME
)

print(
    "Alignment parameters :",
    f"{alignment_parameters:,}"
)

print(
    "Alignment size       :",
    f"{alignment_size_mb:.4f} MB"
)

print(
    "Test KD loss         :",
    f"{float(total_kd_loss.detach().cpu()):.8f}"
)

print()

print(
    "[OK] P3 feature alignment validated."
)

print(
    "[OK] P4 feature alignment validated."
)

print(
    "[OK] P5 feature alignment validated."
)

print(
    "[OK] Feature KD loss validated."
)

print()

print(
    "Results saved to:"
)

print(
    SECTION3_JSON
)

print()

print("=" * 70)
print("[OK] SECTION 3 COMPLETED")
print("=" * 70)

print()

print(
    "Ready for SECTION 4 — "
    "Knowledge Distillation Training."
)

ECOBOT-X KNOWLEDGE DISTILLATION
SECTION 3 — TEACHER / STUDENT FEATURE ALIGNMENT

LOADING YOLOv8n TEACHER

[OK] Teacher loaded.
[OK] Teacher frozen.
Teacher type: DetectionModel

TEACHER FEATURE MAP CANDIDATES
[000] Layer=00 | Conv                 | Shape=(1, 16, 320, 320)
[001] Layer=01 | Conv                 | Shape=(1, 32, 160, 160)
[002] Layer=02 | C2f                  | Shape=(1, 32, 160, 160)
[003] Layer=03 | Conv                 | Shape=(1, 64, 80, 80)
[004] Layer=04 | C2f                  | Shape=(1, 64, 80, 80)
[005] Layer=05 | Conv                 | Shape=(1, 128, 40, 40)
[006] Layer=06 | C2f                  | Shape=(1, 128, 40, 40)
[007] Layer=07 | Conv                 | Shape=(1, 256, 20, 20)
[008] Layer=08 | C2f                  | Shape=(1, 256, 20, 20)
[009] Layer=09 | SPPF                 | Shape=(1, 256, 20, 20)
[010] Layer=10 | Upsample             | Shape=(1, 256, 40, 40)
[011] Layer=11 | Concat               | Shape=(1, 384, 40, 40)
[012] Layer=12 | C2f              

In [5]:
# ======================================================================
# ECOBOT-X KNOWLEDGE DISTILLATION
# SECTION 4 — FEATURE KNOWLEDGE DISTILLATION TRAINING
# ======================================================================
#
# Teacher  : YOLOv8n
# Student  : EcoBotX-Tiny
# KD Type  : Feature Knowledge Distillation
# Features : P3 / P4 / P5
#
# IMPORTANT:
# Run SECTIONS 1, 2 and 3 before running this section.
#
# Dataset:
# G:\EcoBotX_YOLO\images
#
# Training:
# 30 epochs
# GPU-first execution
# ======================================================================

import time
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np


# ======================================================================
# 1. HEADER
# ======================================================================

print()
print("=" * 70)
print("ECOBOT-X KNOWLEDGE DISTILLATION")
print("SECTION 4 — FEATURE KNOWLEDGE DISTILLATION TRAINING")
print("=" * 70)


# ======================================================================
# 2. CHECK PREVIOUS SECTIONS
# ======================================================================

print()
print("=" * 70)
print("CHECKING PREVIOUS SECTIONS")
print("=" * 70)

required_objects = [
    "EcoBotXStudent",
    "FeatureAlign",
    "teacher",
    "teacher_features",
    "TEACHER_CHANNELS",
    "STUDENT_CHANNELS",
    "IMAGE_SIZE",
    "NUM_CLASSES",
    "CLASS_NAMES",
    "DEVICE"
]

for obj_name in required_objects:

    if obj_name in globals():
        print(f"[OK] {obj_name} found.")

    else:
        raise RuntimeError(
            f"[ERROR] {obj_name} not found.\n"
            "Please run Sections 1–3 first."
        )


# ======================================================================
# 3. DEVICE CONFIGURATION
# ======================================================================

print()
print("=" * 70)
print("DEVICE CONFIGURATION")
print("=" * 70)

if torch.cuda.is_available():

    DEVICE = torch.device("cuda")

    print("[OK] CUDA is available.")
    print("GPU :", torch.cuda.get_device_name(0))

    gpu_properties = torch.cuda.get_device_properties(0)

    print(
        "GPU memory :",
        f"{gpu_properties.total_memory / (1024 ** 3):.2f} GB"
    )

else:

    DEVICE = torch.device("cpu")

    print("[WARNING] CUDA unavailable.")
    print("[WARNING] Training will use CPU.")


# ======================================================================
# 4. KD CONFIGURATION
# ======================================================================

KD_EPOCHS = 30

KD_BATCH_SIZE = 4

KD_LEARNING_RATE = 1e-3

KD_WEIGHT_DECAY = 5e-4

KD_NUM_WORKERS = 0

KD_GRAD_CLIP = 10.0

KD_SAVE_PERIOD = 10

KD_FEATURE_WEIGHT = 1.0


print()
print("=" * 70)
print("KD CONFIGURATION")
print("=" * 70)

print("Epochs        :", KD_EPOCHS)
print("Batch size    :", KD_BATCH_SIZE)
print("Learning rate :", KD_LEARNING_RATE)
print("Weight decay  :", KD_WEIGHT_DECAY)
print("Workers       :", KD_NUM_WORKERS)
print("Device        :", DEVICE)


# ======================================================================
# 5. DATASET PATH
# ======================================================================

print()
print("=" * 70)
print("DATASET PATHS")
print("=" * 70)

DATASET_ROOT = Path(
    r"G:\EcoBotX_YOLO\images"
)

TRAIN_IMAGES = DATASET_ROOT / "train"

VAL_IMAGES = DATASET_ROOT / "val"

print("Dataset root      :", DATASET_ROOT)
print("Training images   :", TRAIN_IMAGES)
print("Validation images :", VAL_IMAGES)


# ======================================================================
# 6. DATASET VALIDATION
# ======================================================================

if not DATASET_ROOT.exists():

    raise FileNotFoundError(
        f"Dataset root not found:\n{DATASET_ROOT}"
    )


if not TRAIN_IMAGES.exists():

    raise FileNotFoundError(
        f"Training directory not found:\n{TRAIN_IMAGES}"
    )


if not VAL_IMAGES.exists():

    raise FileNotFoundError(
        f"Validation directory not found:\n{VAL_IMAGES}"
    )


print()
print("[OK] Dataset root found.")
print("[OK] Training directory found.")
print("[OK] Validation directory found.")


# ======================================================================
# 7. CREATE STUDENT
# ======================================================================

print()
print("=" * 70)
print("CREATING ECOBOT-X STUDENT")
print("=" * 70)

student = EcoBotXStudent(
    num_classes=NUM_CLASSES
).to(DEVICE)

print("[OK] Student created.")

print(
    "Student device :",
    next(student.parameters()).device
)


# ======================================================================
# 8. CREATE FEATURE ALIGNMENT
# ======================================================================

print()
print("=" * 70)
print("CREATING FEATURE ALIGNMENT")
print("=" * 70)

alignment = nn.ModuleDict({

    "P3": FeatureAlign(
        TEACHER_CHANNELS["P3"],
        STUDENT_CHANNELS["P3"]
    ),

    "P4": FeatureAlign(
        TEACHER_CHANNELS["P4"],
        STUDENT_CHANNELS["P4"]
    ),

    "P5": FeatureAlign(
        TEACHER_CHANNELS["P5"],
        STUDENT_CHANNELS["P5"]
    )

}).to(DEVICE)

print("[OK] P3 alignment created.")
print("[OK] P4 alignment created.")
print("[OK] P5 alignment created.")


# ======================================================================
# 9. MODEL INFORMATION
# ======================================================================

student_params = sum(
    p.numel()
    for p in student.parameters()
)

trainable_student_params = sum(
    p.numel()
    for p in student.parameters()
    if p.requires_grad
)

alignment_params = sum(
    p.numel()
    for p in alignment.parameters()
)

student_size_mb = (
    student_params * 4
    / (1024 ** 2)
)


print()
print("=" * 70)
print("MODEL INFORMATION")
print("=" * 70)

print(
    "Student parameters       :",
    f"{student_params:,}"
)

print(
    "Trainable student params:",
    f"{trainable_student_params:,}"
)

print(
    "Student FP32 size       :",
    f"{student_size_mb:.3f} MB"
)

print(
    "Alignment parameters    :",
    f"{alignment_params:,}"
)


# ======================================================================
# 10. DATASET CLASS
# ======================================================================

class YOLOImageDataset(Dataset):

    def __init__(
        self,
        image_dir,
        image_size
    ):

        self.image_dir = Path(image_dir)

        self.image_size = image_size

        valid_extensions = {
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".webp"
        }

        self.images = sorted(
            [
                p
                for p in self.image_dir.iterdir()
                if (
                    p.is_file()
                    and
                    p.suffix.lower()
                    in valid_extensions
                )
            ]
        )

        if len(self.images) == 0:

            raise RuntimeError(
                f"No images found in:\n{self.image_dir}"
            )


    def __len__(self):

        return len(self.images)


    def __getitem__(self, index):

        image_path = self.images[index]

        image = Image.open(
            image_path
        ).convert("RGB")

        image = image.resize(
            (
                self.image_size,
                self.image_size
            ),
            Image.Resampling.BILINEAR
        )

        image_array = np.asarray(
            image,
            dtype=np.float32
        )

        image_tensor = (
            torch.from_numpy(
                image_array
            )
            .permute(2, 0, 1)
            / 255.0
        )

        return image_tensor


# ======================================================================
# 11. CREATE DATASETS
# ======================================================================

print()
print("=" * 70)
print("CREATING DATASETS")
print("=" * 70)

train_dataset = YOLOImageDataset(
    TRAIN_IMAGES,
    IMAGE_SIZE
)

val_dataset = YOLOImageDataset(
    VAL_IMAGES,
    IMAGE_SIZE
)

print(
    "Training images   :",
    len(train_dataset)
)

print(
    "Validation images :",
    len(val_dataset)
)


# ======================================================================
# 12. CREATE DATALOADERS
# ======================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=KD_BATCH_SIZE,
    shuffle=True,
    num_workers=KD_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=KD_BATCH_SIZE,
    shuffle=False,
    num_workers=KD_NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

print()
print("[OK] Training DataLoader created.")
print("[OK] Validation DataLoader created.")

print(
    "Training batches   :",
    len(train_loader)
)

print(
    "Validation batches :",
    len(val_loader)
)


# ======================================================================
# 13. TEACHER SETUP
# ======================================================================

print()
print("=" * 70)
print("SETTING UP YOLOv8n TEACHER")
print("=" * 70)

teacher.eval()

for parameter in teacher.parameters():

    parameter.requires_grad = False


print("[OK] Teacher set to evaluation mode.")
print("[OK] Teacher parameters frozen.")


# ======================================================================
# 14. TEACHER FEATURE LOCATIONS
# ======================================================================

P3_LAYER = int(
    teacher_features["P3"]["layer_index"]
)

P4_LAYER = int(
    teacher_features["P4"]["layer_index"]
)

P5_LAYER = int(
    teacher_features["P5"]["layer_index"]
)

print()
print("=" * 70)
print("TEACHER FEATURE LOCATIONS")
print("=" * 70)

print(
    f"P3 : Layer {P3_LAYER} | "
    f"Channels {TEACHER_CHANNELS['P3']} | "
    f"80x80"
)

print(
    f"P4 : Layer {P4_LAYER} | "
    f"Channels {TEACHER_CHANNELS['P4']} | "
    f"40x40"
)

print(
    f"P5 : Layer {P5_LAYER} | "
    f"Channels {TEACHER_CHANNELS['P5']} | "
    f"20x20"
)


# ======================================================================
# 15. TEACHER FEATURE EXTRACTION
# ======================================================================

def get_teacher_features(images):

    captured = {}


    # --------------------------------------------------------------
    # P3
    # --------------------------------------------------------------

    def hook_p3(
        module,
        inputs,
        output
    ):

        if torch.is_tensor(output):

            captured["P3"] = output


    # --------------------------------------------------------------
    # P4
    # --------------------------------------------------------------

    def hook_p4(
        module,
        inputs,
        output
    ):

        if torch.is_tensor(output):

            captured["P4"] = output


    # --------------------------------------------------------------
    # P5
    # --------------------------------------------------------------

    def hook_p5(
        module,
        inputs,
        output
    ):

        if torch.is_tensor(output):

            captured["P5"] = output


    # --------------------------------------------------------------
    # REGISTER HOOKS
    # --------------------------------------------------------------

    hook1 = teacher.model[
        P3_LAYER
    ].register_forward_hook(
        hook_p3
    )

    hook2 = teacher.model[
        P4_LAYER
    ].register_forward_hook(
        hook_p4
    )

    hook3 = teacher.model[
        P5_LAYER
    ].register_forward_hook(
        hook_p5
    )


    try:

        with torch.no_grad():

            _ = teacher(images)

    finally:

        hook1.remove()
        hook2.remove()
        hook3.remove()


    # --------------------------------------------------------------
    # VERIFY
    # --------------------------------------------------------------

    for feature_name in ["P3", "P4", "P5"]:

        if feature_name not in captured:

            raise RuntimeError(
                f"Teacher {feature_name} "
                "feature was not captured."
            )


    # --------------------------------------------------------------
    # CLONE NORMAL TENSORS
    # --------------------------------------------------------------

    captured["P3"] = (
        captured["P3"]
        .detach()
        .clone()
    )

    captured["P4"] = (
        captured["P4"]
        .detach()
        .clone()
    )

    captured["P5"] = (
        captured["P5"]
        .detach()
        .clone()
    )

    return captured


# ======================================================================
# 16. FEATURE KD LOSS
# ======================================================================

def kd_feature_loss(
    student_feature,
    teacher_feature
):

    # --------------------------------------------------------------
    # Spatial alignment
    # --------------------------------------------------------------

    if (
        student_feature.shape[2:]
        !=
        teacher_feature.shape[2:]
    ):

        teacher_feature = F.interpolate(
            teacher_feature,
            size=student_feature.shape[-2:],
            mode="bilinear",
            align_corners=False
        )


    # --------------------------------------------------------------
    # Channel validation
    # --------------------------------------------------------------

    if (
        student_feature.shape[1]
        !=
        teacher_feature.shape[1]
    ):

        raise RuntimeError(
            "\nFeature channel mismatch:\n"
            f"Student channels: "
            f"{student_feature.shape[1]}\n"
            f"Teacher channels: "
            f"{teacher_feature.shape[1]}"
        )


    # --------------------------------------------------------------
    # L2 NORMALIZATION
    # --------------------------------------------------------------

    student_norm = F.normalize(
        student_feature,
        p=2,
        dim=1
    )

    teacher_norm = F.normalize(
        teacher_feature,
        p=2,
        dim=1
    )


    # --------------------------------------------------------------
    # MSE
    # --------------------------------------------------------------

    return F.mse_loss(
        student_norm,
        teacher_norm
    )


# ======================================================================
# 17. OPTIMIZER
# ======================================================================

optimizer = torch.optim.AdamW(
    [
        {
            "params": student.parameters(),
            "lr": KD_LEARNING_RATE
        },
        {
            "params": alignment.parameters(),
            "lr": KD_LEARNING_RATE
        }
    ],
    weight_decay=KD_WEIGHT_DECAY
)


# ======================================================================
# 18. LEARNING RATE SCHEDULER
# ======================================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=KD_EPOCHS,
    eta_min=KD_LEARNING_RATE * 0.01
)


print()
print("[OK] AdamW optimizer created.")
print("[OK] CosineAnnealingLR created.")


# ======================================================================
# 19. INITIAL FORWARD TEST
# ======================================================================

print()
print("=" * 70)
print("INITIAL KD FORWARD TEST")
print("=" * 70)

images = next(
    iter(train_loader)
)

print(
    "CPU batch shape :",
    tuple(images.shape)
)


# ----------------------------------------------------------------------
# MOVE IMAGE TO GPU
# ----------------------------------------------------------------------

images = images.to(
    DEVICE,
    non_blocking=True
)

print(
    "Device          :",
    images.device
)


# ======================================================================
# 20. STUDENT TEST
# ======================================================================

student.eval()
alignment.eval()

with torch.no_grad():

    student_output = student(
        images
    )


print()
print("[OK] Student forward completed.")

print(
    "Student P3:",
    tuple(student_output["P3"].shape)
)

print(
    "Student P4:",
    tuple(student_output["P4"].shape)
)

print(
    "Student P5:",
    tuple(student_output["P5"].shape)
)


# ======================================================================
# 21. TEACHER TEST
# ======================================================================

teacher_output = get_teacher_features(
    images
)

print()
print("[OK] Teacher forward completed.")

print(
    "Teacher P3:",
    tuple(teacher_output["P3"].shape)
)

print(
    "Teacher P4:",
    tuple(teacher_output["P4"].shape)
)

print(
    "Teacher P5:",
    tuple(teacher_output["P5"].shape)
)


# ======================================================================
# 22. ALIGNMENT TEST
# ======================================================================

with torch.no_grad():

    aligned_p3 = alignment["P3"](
        teacher_output["P3"]
    )

    aligned_p4 = alignment["P4"](
        teacher_output["P4"]
    )

    aligned_p5 = alignment["P5"](
        teacher_output["P5"]
    )


print()
print("[OK] Feature alignment completed.")

print(
    "Aligned P3:",
    tuple(aligned_p3.shape)
)

print(
    "Aligned P4:",
    tuple(aligned_p4.shape)
)

print(
    "Aligned P5:",
    tuple(aligned_p5.shape)
)


# ======================================================================
# 23. INITIAL KD LOSS
# ======================================================================

loss_p3 = kd_feature_loss(
    student_output["P3"],
    aligned_p3
)

loss_p4 = kd_feature_loss(
    student_output["P4"],
    aligned_p4
)

loss_p5 = kd_feature_loss(
    student_output["P5"],
    aligned_p5
)

initial_kd_loss = (
    loss_p3
    +
    loss_p4
    +
    loss_p5
) / 3.0


print()
print("=" * 70)
print("INITIAL KD TEST RESULT")
print("=" * 70)

print(
    "P3 KD loss    :",
    f"{loss_p3.item():.8f}"
)

print(
    "P4 KD loss    :",
    f"{loss_p4.item():.8f}"
)

print(
    "P5 KD loss    :",
    f"{loss_p5.item():.8f}"
)

print(
    "Total KD loss :",
    f"{initial_kd_loss.item():.8f}"
)

print()
print("[OK] Initial KD test passed.")


# ======================================================================
# 24. CLEAN TEST MEMORY
# ======================================================================

del images
del student_output
del teacher_output
del aligned_p3
del aligned_p4
del aligned_p5

del loss_p3
del loss_p4
del loss_p5
del initial_kd_loss

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ======================================================================
# 25. OUTPUT DIRECTORY
# ======================================================================

KD_OUTPUT_DIR = (
    Path(
        r"G:\EcoBotX_YOLO_training"
    )
    /
    "knowledge_distillation"
    /
    "section4_feature_kd"
)

KD_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print()
print("Output directory:")
print(KD_OUTPUT_DIR)


# ======================================================================
# 26. TRAINING HISTORY
# ======================================================================

history = {

    "epoch": [],

    "train_loss": [],
    "val_loss": [],

    "train_p3": [],
    "train_p4": [],
    "train_p5": [],

    "val_p3": [],
    "val_p4": [],
    "val_p5": [],

    "learning_rate": [],

    "epoch_time": []
}


# ======================================================================
# 27. CHECKPOINT FUNCTION
# ======================================================================

def save_checkpoint(
    path,
    epoch,
    best_val_loss
):

    torch.save(
        {
            "epoch":
                epoch,

            "student_state_dict":
                student.state_dict(),

            "alignment_state_dict":
                alignment.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "best_val_loss":
                best_val_loss,

            "teacher_channels":
                TEACHER_CHANNELS,

            "student_channels":
                STUDENT_CHANNELS,

            "class_names":
                CLASS_NAMES,

            "image_size":
                IMAGE_SIZE,

            "kd_epochs":
                KD_EPOCHS,

            "kd_learning_rate":
                KD_LEARNING_RATE,

            "history":
                history
        },
        path
    )


# ======================================================================
# 28. TRAINING START
# ======================================================================

print()
print("=" * 70)
print("STARTING FEATURE KNOWLEDGE DISTILLATION")
print("=" * 70)

print()
print("Teacher : YOLOv8n")
print("Student : EcoBotX-Tiny")
print("Epochs  :", KD_EPOCHS)
print("Device  :", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU     :",
        torch.cuda.get_device_name(0)
    )


best_val_loss = float("inf")

training_start = time.time()


# ======================================================================
# 29. EPOCH LOOP
# ======================================================================

for epoch in range(
    1,
    KD_EPOCHS + 1
):

    epoch_start = time.time()


    # ==============================================================
    # TRAIN MODE
    # ==============================================================

    student.train()
    alignment.train()
    teacher.eval()


    train_loss = 0.0

    train_p3 = 0.0
    train_p4 = 0.0
    train_p5 = 0.0

    train_batches = 0


    # ==============================================================
    # TRAINING
    # ==============================================================

    for batch_index, images in enumerate(
        train_loader,
        start=1
    ):

        # ----------------------------------------------------------
        # GPU TRANSFER
        # ----------------------------------------------------------

        images = images.to(
            DEVICE,
            non_blocking=True
        )


        # ----------------------------------------------------------
        # CLEAR GRADIENTS
        # ----------------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------------
        # STUDENT
        # ----------------------------------------------------------

        student_output = student(
            images
        )


        # ----------------------------------------------------------
        # TEACHER
        # ----------------------------------------------------------

        teacher_output = get_teacher_features(
            images
        )


        # ----------------------------------------------------------
        # FEATURE ALIGNMENT
        # ----------------------------------------------------------

        aligned_p3 = alignment["P3"](
            teacher_output["P3"]
        )

        aligned_p4 = alignment["P4"](
            teacher_output["P4"]
        )

        aligned_p5 = alignment["P5"](
            teacher_output["P5"]
        )


        # ----------------------------------------------------------
        # KD LOSS
        # ----------------------------------------------------------

        loss_p3 = kd_feature_loss(
            student_output["P3"],
            aligned_p3
        )

        loss_p4 = kd_feature_loss(
            student_output["P4"],
            aligned_p4
        )

        loss_p5 = kd_feature_loss(
            student_output["P5"],
            aligned_p5
        )


        # ----------------------------------------------------------
        # TOTAL KD LOSS
        # ----------------------------------------------------------

        kd_loss = (
            loss_p3
            +
            loss_p4
            +
            loss_p5
        ) / 3.0


        total_loss = (
            KD_FEATURE_WEIGHT
            *
            kd_loss
        )


        # ----------------------------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------------------------

        total_loss.backward()


        # ----------------------------------------------------------
        # GRADIENT CLIPPING
        # ----------------------------------------------------------

        torch.nn.utils.clip_grad_norm_(
            list(student.parameters())
            +
            list(alignment.parameters()),
            KD_GRAD_CLIP
        )


        # ----------------------------------------------------------
        # OPTIMIZER STEP
        # ----------------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------------
        # ACCUMULATE
        # ----------------------------------------------------------

        train_loss += (
            total_loss.detach().item()
        )

        train_p3 += (
            loss_p3.detach().item()
        )

        train_p4 += (
            loss_p4.detach().item()
        )

        train_p5 += (
            loss_p5.detach().item()
        )

        train_batches += 1


        # ----------------------------------------------------------
        # PROGRESS
        # ----------------------------------------------------------

        if (
            batch_index == 1
            or
            batch_index % 50 == 0
            or
            batch_index == len(train_loader)
        ):

            print(
                f"\r"
                f"Epoch "
                f"{epoch:02d}/{KD_EPOCHS} | "
                f"Batch "
                f"{batch_index:04d}/"
                f"{len(train_loader)} | "
                f"KD Loss "
                f"{total_loss.item():.6f}",
                end="",
                flush=True
            )


    print()


    # ==============================================================
    # VALIDATION
    # ==============================================================

    student.eval()
    alignment.eval()
    teacher.eval()


    val_loss = 0.0

    val_p3 = 0.0
    val_p4 = 0.0
    val_p5 = 0.0

    val_batches = 0


    with torch.no_grad():

        for images in val_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )


            # ------------------------------------------------------
            # STUDENT
            # ------------------------------------------------------

            student_output = student(
                images
            )


            # ------------------------------------------------------
            # TEACHER
            # ------------------------------------------------------

            teacher_output = get_teacher_features(
                images
            )


            # ------------------------------------------------------
            # ALIGNMENT
            # ------------------------------------------------------

            aligned_p3 = alignment["P3"](
                teacher_output["P3"]
            )

            aligned_p4 = alignment["P4"](
                teacher_output["P4"]
            )

            aligned_p5 = alignment["P5"](
                teacher_output["P5"]
            )


            # ------------------------------------------------------
            # LOSS
            # ------------------------------------------------------

            loss_p3 = kd_feature_loss(
                student_output["P3"],
                aligned_p3
            )

            loss_p4 = kd_feature_loss(
                student_output["P4"],
                aligned_p4
            )

            loss_p5 = kd_feature_loss(
                student_output["P5"],
                aligned_p5
            )


            batch_loss = (
                loss_p3
                +
                loss_p4
                +
                loss_p5
            ) / 3.0


            val_loss += (
                batch_loss.item()
            )

            val_p3 += (
                loss_p3.item()
            )

            val_p4 += (
                loss_p4.item()
            )

            val_p5 += (
                loss_p5.item()
            )

            val_batches += 1


    # ==============================================================
    # AVERAGES
    # ==============================================================

    avg_train_loss = (
        train_loss
        /
        max(train_batches, 1)
    )

    avg_train_p3 = (
        train_p3
        /
        max(train_batches, 1)
    )

    avg_train_p4 = (
        train_p4
        /
        max(train_batches, 1)
    )

    avg_train_p5 = (
        train_p5
        /
        max(train_batches, 1)
    )


    avg_val_loss = (
        val_loss
        /
        max(val_batches, 1)
    )

    avg_val_p3 = (
        val_p3
        /
        max(val_batches, 1)
    )

    avg_val_p4 = (
        val_p4
        /
        max(val_batches, 1)
    )

    avg_val_p5 = (
        val_p5
        /
        max(val_batches, 1)
    )


    # ==============================================================
    # LR SCHEDULER
    # ==============================================================

    scheduler.step()

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )


    # ==============================================================
    # EPOCH TIME
    # ==============================================================

    epoch_time = (
        time.time()
        -
        epoch_start
    )


    # ==============================================================
    # HISTORY
    # ==============================================================

    history["epoch"].append(
        epoch
    )

    history["train_loss"].append(
        avg_train_loss
    )

    history["val_loss"].append(
        avg_val_loss
    )

    history["train_p3"].append(
        avg_train_p3
    )

    history["train_p4"].append(
        avg_train_p4
    )

    history["train_p5"].append(
        avg_train_p5
    )

    history["val_p3"].append(
        avg_val_p3
    )

    history["val_p4"].append(
        avg_val_p4
    )

    history["val_p5"].append(
        avg_val_p5
    )

    history["learning_rate"].append(
        current_lr
    )

    history["epoch_time"].append(
        epoch_time
    )


    # ==============================================================
    # EPOCH SUMMARY
    # ==============================================================

    print()
    print("-" * 70)

    print(
        f"Epoch {epoch:02d}/{KD_EPOCHS}"
    )

    print(
        f"Train KD Loss : "
        f"{avg_train_loss:.8f}"
    )

    print(
        f"Val KD Loss   : "
        f"{avg_val_loss:.8f}"
    )

    print(
        f"Val P3 Loss   : "
        f"{avg_val_p3:.8f}"
    )

    print(
        f"Val P4 Loss   : "
        f"{avg_val_p4:.8f}"
    )

    print(
        f"Val P5 Loss   : "
        f"{avg_val_p5:.8f}"
    )

    print(
        f"Learning Rate : "
        f"{current_lr:.8f}"
    )

    print(
        f"Epoch Time    : "
        f"{epoch_time:.2f} sec"
    )


    # ==============================================================
    # GPU MEMORY
    # ==============================================================

    if torch.cuda.is_available():

        allocated = (
            torch.cuda.memory_allocated()
            /
            (1024 ** 3)
        )

        reserved = (
            torch.cuda.memory_reserved()
            /
            (1024 ** 3)
        )

        print(
            f"GPU Memory    : "
            f"{allocated:.2f} GB allocated | "
            f"{reserved:.2f} GB reserved"
        )


    print("-" * 70)


    # ==============================================================
    # BEST MODEL
    # ==============================================================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        best_path = (
            KD_OUTPUT_DIR
            /
            "best_student_kd.pt"
        )

        save_checkpoint(
            best_path,
            epoch,
            best_val_loss
        )

        print()
        print("[BEST MODEL SAVED]")
        print(best_path)


    # ==============================================================
    # PERIODIC CHECKPOINT
    # ==============================================================

    if (
        epoch % KD_SAVE_PERIOD == 0
        or
        epoch == KD_EPOCHS
    ):

        checkpoint_path = (
            KD_OUTPUT_DIR
            /
            f"student_kd_epoch_{epoch:03d}.pt"
        )

        save_checkpoint(
            checkpoint_path,
            epoch,
            best_val_loss
        )

        print()
        print("[CHECKPOINT SAVED]")
        print(checkpoint_path)


    # ==============================================================
    # CLEAN GPU CACHE
    # ==============================================================

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ======================================================================
# 30. TRAINING COMPLETE
# ======================================================================

total_training_time = (
    time.time()
    -
    training_start
)

print()
print("=" * 70)
print("KD TRAINING COMPLETED")
print("=" * 70)

print()

print(
    "Epochs completed :",
    KD_EPOCHS
)

print(
    "Best validation KD loss :",
    f"{best_val_loss:.8f}"
)

print(
    "Total training time :",
    f"{total_training_time / 3600:.2f} hours"
)


# ======================================================================
# 31. SAVE FINAL STUDENT
# ======================================================================

final_student_path = (
    KD_OUTPUT_DIR
    /
    "final_student_kd.pt"
)

torch.save(
    {
        "student_state_dict":
            student.state_dict(),

        "alignment_state_dict":
            alignment.state_dict(),

        "teacher_channels":
            TEACHER_CHANNELS,

        "student_channels":
            STUDENT_CHANNELS,

        "best_val_loss":
            best_val_loss,

        "class_names":
            CLASS_NAMES,

        "image_size":
            IMAGE_SIZE,

        "kd_epochs":
            KD_EPOCHS,

        "kd_learning_rate":
            KD_LEARNING_RATE,

        "history":
            history
    },
    final_student_path
)


# ======================================================================
# 32. SAVE TRAINING HISTORY
# ======================================================================

history_path = (
    KD_OUTPUT_DIR
    /
    "training_history.json"
)

with open(
    history_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        history,
        f,
        indent=2
    )


# ======================================================================
# 33. FINAL OUTPUT
# ======================================================================

print()
print("=" * 70)
print("SECTION 4 OUTPUT")
print("=" * 70)

print()

print(
    "Best KD student:"
)

print(
    KD_OUTPUT_DIR
    /
    "best_student_kd.pt"
)

print()

print(
    "Final KD student:"
)

print(
    final_student_path
)

print()

print(
    "Training history:"
)

print(
    history_path
)

print()
print("=" * 70)
print("[OK] SECTION 4 COMPLETED SUCCESSFULLY")
print("=" * 70)


ECOBOT-X KNOWLEDGE DISTILLATION
SECTION 4 — FEATURE KNOWLEDGE DISTILLATION TRAINING

CHECKING PREVIOUS SECTIONS
[OK] EcoBotXStudent found.
[OK] FeatureAlign found.
[OK] teacher found.
[OK] teacher_features found.
[OK] TEACHER_CHANNELS found.
[OK] STUDENT_CHANNELS found.
[OK] IMAGE_SIZE found.
[OK] NUM_CLASSES found.
[OK] CLASS_NAMES found.
[OK] DEVICE found.

DEVICE CONFIGURATION
[OK] CUDA is available.
GPU : NVIDIA GeForce RTX 3050 Laptop GPU
GPU memory : 4.00 GB

KD CONFIGURATION
Epochs        : 30
Batch size    : 4
Learning rate : 0.001
Weight decay  : 0.0005
Workers       : 0
Device        : cuda

DATASET PATHS
Dataset root      : G:\EcoBotX_YOLO\images
Training images   : G:\EcoBotX_YOLO\images\train
Validation images : G:\EcoBotX_YOLO\images\val

[OK] Dataset root found.
[OK] Training directory found.
[OK] Validation directory found.

CREATING ECOBOT-X STUDENT
[OK] Student created.
Student device : cuda:0

CREATING FEATURE ALIGNMENT
[OK] P3 alignment created.
[OK] P4 alignment c

In [13]:
# ======================================================================
# ECOBOT-X KNOWLEDGE DISTILLATION
# SECTION 5 — DETECTION HEAD + SUPERVISED FINE-TUNING
# ======================================================================
#
# EXACT STUDENT ARCHITECTURE:
#   Backbone : GhostConv + DWConv + Coordinate Attention
#   Neck     : Lightweight Ghost-PAN
#   Features : P3 / P4 / P5
#   Channels : 64 / 96 / 128
#
# INITIALIZATION:
#   Section 4 feature-KD student checkpoint
#
# TRAINING:
#   Supervised detection fine-tuning
#   30 epochs
#   GPU only
#
# DATASET:
#   G:\EcoBotX_YOLO
#
# ======================================================================

import os
import json
import time
import math
import random
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("ECOBOT-X KNOWLEDGE DISTILLATION")
print("SECTION 5 — DETECTION HEAD + SUPERVISED FINE-TUNING")
print("=" * 70)

print()
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
    print("CUDA   :", torch.version.cuda)


# ======================================================================
# 2. PATHS
# ======================================================================

DATASET_ROOT = Path(r"G:\EcoBotX_YOLO")

TRAIN_IMAGES = DATASET_ROOT / "images" / "train"
TRAIN_LABELS = DATASET_ROOT / "labels" / "train"

VAL_IMAGES = DATASET_ROOT / "images" / "val"
VAL_LABELS = DATASET_ROOT / "labels" / "val"

TEST_IMAGES = DATASET_ROOT / "images" / "test"
TEST_LABELS = DATASET_ROOT / "labels" / "test"

KD_CHECKPOINT = Path(
    r"G:\EcoBotX_YOLO_training\knowledge_distillation"
    r"\section4_feature_kd\best_student_kd.pt"
)

OUTPUT_ROOT = Path(
    r"G:\EcoBotX_YOLO_training\knowledge_distillation"
    r"\section5_supervised_finetuning"
)

BEST_DIR = OUTPUT_ROOT / "best"
FINAL_DIR = OUTPUT_ROOT / "final"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"

BEST_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================================
# 3. TRAINING CONFIGURATION
# ======================================================================

IMAGE_SIZE = 640

NUM_CLASSES = 4

CLASS_NAMES = [
    "BOTTLE",
    "CAN",
    "PAPER",
    "WRAPPER"
]

EPOCHS = 30

BATCH_SIZE = 8

LEARNING_RATE = 1e-3

BACKBONE_LR = 1e-4

WEIGHT_DECAY = 5e-4

NUM_WORKERS = 0

PIN_MEMORY = True

CONF_THRESHOLD = 0.25

IOU_THRESHOLD = 0.50

SAVE_PERIOD = 5


# ======================================================================
# 4. PATH VALIDATION
# ======================================================================

print()
print("=" * 70)
print("PATH VALIDATION")
print("=" * 70)

required_paths = {
    "Train images": TRAIN_IMAGES,
    "Train labels": TRAIN_LABELS,
    "Validation images": VAL_IMAGES,
    "Validation labels": VAL_LABELS,
    "Test images": TEST_IMAGES,
    "Test labels": TEST_LABELS,
    "KD checkpoint": KD_CHECKPOINT,
}

for name, path in required_paths.items():

    if path.exists():
        print(f"[OK] {name}: {path}")

    else:
        raise FileNotFoundError(
            f"\n[MISSING] {name}\n"
            f"Path: {path}\n"
        )


# ======================================================================
# 5. GHOST CONV
# ======================================================================

class GhostConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=1,
        stride=1,
        activation=True
    ):

        super().__init__()

        padding = kernel_size // 2

        primary_channels = (
            out_channels + 1
        ) // 2

        cheap_channels = (
            out_channels -
            primary_channels
        )

        self.primary = nn.Sequential(

            nn.Conv2d(
                in_channels,
                primary_channels,
                kernel_size,
                stride,
                padding,
                bias=False
            ),

            nn.BatchNorm2d(
                primary_channels
            ),

            nn.ReLU6(
                inplace=True
            )
        )

        self.cheap = None

        if cheap_channels > 0:

            groups = (
                primary_channels
                if cheap_channels % primary_channels == 0
                else 1
            )

            self.cheap = nn.Sequential(

                nn.Conv2d(
                    primary_channels,
                    cheap_channels,
                    kernel_size=3,
                    stride=1,
                    padding=1,
                    groups=groups,
                    bias=False
                ),

                nn.BatchNorm2d(
                    cheap_channels
                ),

                nn.ReLU6(
                    inplace=True
                )
            )

        self.out_channels = out_channels

    def forward(self, x):

        y = self.primary(x)

        if self.cheap is None:
            return y

        z = self.cheap(y)

        return torch.cat(
            [y, z],
            dim=1
        )


# ======================================================================
# 6. DEPTHWISE SEPARABLE CONVOLUTION
# ======================================================================

class DWConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):

        super().__init__()

        self.depthwise = nn.Sequential(

            nn.Conv2d(
                in_channels,
                in_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=in_channels,
                bias=False
            ),

            nn.BatchNorm2d(
                in_channels
            ),

            nn.ReLU6(
                inplace=True
            )
        )

        self.pointwise = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU6(
                inplace=True
            )
        )

    def forward(self, x):

        x = self.depthwise(x)
        x = self.pointwise(x)

        return x


# ======================================================================
# 7. COORDINATE ATTENTION
# ======================================================================

class CoordinateAttention(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16
    ):

        super().__init__()

        reduced_channels = max(
            8,
            channels // reduction
        )

        self.conv1 = nn.Conv2d(
            channels,
            reduced_channels,
            kernel_size=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(
            reduced_channels
        )

        self.act = nn.ReLU6(
            inplace=True
        )

        self.conv_h = nn.Conv2d(
            reduced_channels,
            channels,
            kernel_size=1
        )

        self.conv_w = nn.Conv2d(
            reduced_channels,
            channels,
            kernel_size=1
        )

    def forward(self, x):

        identity = x

        b, c, h, w = x.shape

        x_h = x.mean(
            dim=3,
            keepdim=True
        )

        x_w = x.mean(
            dim=2,
            keepdim=True
        )

        x_w = x_w.permute(
            0,
            1,
            3,
            2
        )

        y = torch.cat(
            [x_h, x_w],
            dim=2
        )

        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        y_h, y_w = torch.split(
            y,
            [h, w],
            dim=2
        )

        y_w = y_w.permute(
            0,
            1,
            3,
            2
        )

        attention_h = torch.sigmoid(
            self.conv_h(y_h)
        )

        attention_w = torch.sigmoid(
            self.conv_w(y_w)
        )

        return (
            identity
            * attention_h
            * attention_w
        )


# ======================================================================
# 8. ECOBLOCK
# ======================================================================

class EcoBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1,
        attention=False
    ):

        super().__init__()

        self.ghost = GhostConv(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=1
        )

        self.dwconv = DWConv(
            out_channels,
            out_channels,
            stride=stride
        )

        if attention:

            self.attention = CoordinateAttention(
                out_channels
            )

        else:

            self.attention = nn.Identity()

        if (
            stride != 1
            or
            in_channels != out_channels
        ):

            self.shortcut = nn.Sequential(

                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),

                nn.BatchNorm2d(
                    out_channels
                )
            )

        else:

            self.shortcut = nn.Identity()

        self.act = nn.ReLU6(
            inplace=True
        )

    def forward(self, x):

        identity = self.shortcut(x)

        y = self.ghost(x)

        y = self.dwconv(y)

        y = self.attention(y)

        y = y + identity

        y = self.act(y)

        return y


# ======================================================================
# 9. ECOBOT-X BACKBONE
# ======================================================================

class EcoBotXBackbone(nn.Module):

    def __init__(self):

        super().__init__()

        self.stem = GhostConv(
            3,
            32,
            kernel_size=3,
            stride=2
        )

        self.stage1 = EcoBlock(
            32,
            48,
            stride=2,
            attention=False
        )

        self.stage2 = EcoBlock(
            48,
            64,
            stride=2,
            attention=False
        )

        self.stage2_refine = EcoBlock(
            64,
            64,
            stride=1,
            attention=False
        )

        self.stage3 = EcoBlock(
            64,
            96,
            stride=2,
            attention=True
        )

        self.stage3_refine = EcoBlock(
            96,
            96,
            stride=1,
            attention=True
        )

        self.stage4 = EcoBlock(
            96,
            128,
            stride=2,
            attention=True
        )

        self.stage4_refine = EcoBlock(
            128,
            128,
            stride=1,
            attention=True
        )

    def forward(self, x):

        x = self.stem(x)

        x = self.stage1(x)

        x = self.stage2(x)

        p3 = self.stage2_refine(x)

        x = self.stage3(p3)

        p4 = self.stage3_refine(x)

        x = self.stage4(p4)

        p5 = self.stage4_refine(x)

        return p3, p4, p5


# ======================================================================
# 10. GHOST-PAN
# ======================================================================

class GhostPAN(nn.Module):

    def __init__(self):

        super().__init__()

        self.p5_to_p4 = GhostConv(
            128,
            96,
            kernel_size=1
        )

        self.p4_fuse = GhostConv(
            192,
            96,
            kernel_size=1
        )

        self.p4_to_p3 = GhostConv(
            96,
            64,
            kernel_size=1
        )

        self.p3_fuse = GhostConv(
            128,
            64,
            kernel_size=1
        )

        self.p3_down = DWConv(
            64,
            96,
            stride=2
        )

        self.p4_pan = GhostConv(
            192,
            96,
            kernel_size=1
        )

        self.p4_down = DWConv(
            96,
            128,
            stride=2
        )

        self.p5_pan = GhostConv(
            256,
            128,
            kernel_size=1
        )

    def forward(
        self,
        p3,
        p4,
        p5
    ):

        p5_td = self.p5_to_p4(p5)

        p5_td = F.interpolate(
            p5_td,
            size=p4.shape[-2:],
            mode="nearest"
        )

        p4_td = torch.cat(
            [p4, p5_td],
            dim=1
        )

        p4_td = self.p4_fuse(
            p4_td
        )

        p4_up = self.p4_to_p3(
            p4_td
        )

        p4_up = F.interpolate(
            p4_up,
            size=p3.shape[-2:],
            mode="nearest"
        )

        p3_td = torch.cat(
            [p3, p4_up],
            dim=1
        )

        p3_out = self.p3_fuse(
            p3_td
        )

        p3_down = self.p3_down(
            p3_out
        )

        p4_out = torch.cat(
            [p3_down, p4_td],
            dim=1
        )

        p4_out = self.p4_pan(
            p4_out
        )

        p4_down = self.p4_down(
            p4_out
        )

        p5_out = torch.cat(
            [p4_down, p5],
            dim=1
        )

        p5_out = self.p5_pan(
            p5_out
        )

        return (
            p3_out,
            p4_out,
            p5_out
        )


# ======================================================================
# 11. EXACT SECTION 2 STUDENT
# ======================================================================

class EcoBotXStudent(nn.Module):

    def __init__(
        self,
        num_classes=NUM_CLASSES
    ):

        super().__init__()

        self.num_classes = num_classes

        self.backbone = EcoBotXBackbone()

        self.neck = GhostPAN()

    def forward(self, x):

        p3, p4, p5 = self.backbone(x)

        p3, p4, p5 = self.neck(
            p3,
            p4,
            p5
        )

        return {
            "P3": p3,
            "P4": p4,
            "P5": p5,
            "features": [
                p3,
                p4,
                p5
            ]
        }


# ======================================================================
# 12. ANCHOR-FREE DETECTION HEAD
# ======================================================================
#
# Each location predicts:
#
#   4 box distances
#   4 class logits
#   1 centerness score
#
# Total = 9 outputs/location
#
# P3 stride = 8
# P4 stride = 16
# P5 stride = 32
#
# ======================================================================

class AnchorFreeHead(nn.Module):

    def __init__(
        self,
        in_channels,
        num_classes
    ):

        super().__init__()

        self.num_classes = num_classes

        hidden = max(
            32,
            in_channels // 2
        )

        self.stem = nn.Sequential(

            nn.Conv2d(
                in_channels,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                hidden
            ),

            nn.ReLU6(
                inplace=True
            ),

            nn.Conv2d(
                hidden,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                hidden
            ),

            nn.ReLU6(
                inplace=True
            )
        )

        self.box = nn.Conv2d(
            hidden,
            4,
            kernel_size=1
        )

        self.cls = nn.Conv2d(
            hidden,
            num_classes,
            kernel_size=1
        )

        self.center = nn.Conv2d(
            hidden,
            1,
            kernel_size=1
        )

    def forward(self, x):

        x = self.stem(x)

        box = F.relu(
            self.box(x)
        )

        cls = self.cls(x)

        center = self.center(x)

        return {
            "box": box,
            "cls": cls,
            "center": center
        }


# ======================================================================
# 13. COMPLETE DETECTOR
# ======================================================================

class EcoBotXTinyKDDetector(nn.Module):

    def __init__(
        self,
        num_classes=NUM_CLASSES
    ):

        super().__init__()

        self.num_classes = num_classes

        self.student = EcoBotXStudent(
            num_classes=num_classes
        )

        self.head_p3 = AnchorFreeHead(
            64,
            num_classes
        )

        self.head_p4 = AnchorFreeHead(
            96,
            num_classes
        )

        self.head_p5 = AnchorFreeHead(
            128,
            num_classes
        )

    def forward(self, x):

        features = self.student(x)

        p3 = features["P3"]
        p4 = features["P4"]
        p5 = features["P5"]

        return {
            "P3": self.head_p3(p3),
            "P4": self.head_p4(p4),
            "P5": self.head_p5(p5)
        }


# ======================================================================
# 14. DATASET
# ======================================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


class YOLODetectionDataset(Dataset):

    def __init__(
        self,
        image_dir,
        label_dir,
        image_size=640
    ):

        self.image_dir = Path(image_dir)

        self.label_dir = Path(label_dir)

        self.image_size = image_size

        self.images = sorted(
            [
                p for p in self.image_dir.iterdir()
                if p.suffix.lower() in IMAGE_EXTENSIONS
            ]
        )

        if len(self.images) == 0:

            raise RuntimeError(
                f"No images found in {self.image_dir}"
            )

    def __len__(self):

        return len(self.images)

    def __getitem__(self, index):

        image_path = self.images[index]

        label_path = (
            self.label_dir /
            f"{image_path.stem}.txt"
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        original_w, original_h = image.size

        image = image.resize(
            (
                self.image_size,
                self.image_size
            ),
            Image.Resampling.BILINEAR
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        image = torch.from_numpy(
            image
        ).permute(
            2,
            0,
            1
        )

        targets = []

        if label_path.exists():

            with open(
                label_path,
                "r"
            ) as f:

                for line in f:

                    parts = line.strip().split()

                    if len(parts) != 5:
                        continue

                    cls = int(
                        float(parts[0])
                    )

                    xc = float(parts[1])
                    yc = float(parts[2])
                    w = float(parts[3])
                    h = float(parts[4])

                    if (
                        cls < 0
                        or cls >= NUM_CLASSES
                    ):
                        continue

                    targets.append(
                        [
                            cls,
                            xc,
                            yc,
                            w,
                            h
                        ]
                    )

        if len(targets) == 0:

            targets_tensor = torch.zeros(
                (0, 5),
                dtype=torch.float32
            )

        else:

            targets_tensor = torch.tensor(
                targets,
                dtype=torch.float32
            )

        return image, targets_tensor, str(image_path)


def collate_fn(batch):

    images = torch.stack(
        [
            item[0]
            for item in batch
        ]
    )

    targets = [
        item[1]
        for item in batch
    ]

    paths = [
        item[2]
        for item in batch
    ]

    return images, targets, paths


# ======================================================================
# 15. CREATE DATASETS
# ======================================================================

print()
print("=" * 70)
print("LOADING DATASETS")
print("=" * 70)

train_dataset = YOLODetectionDataset(
    TRAIN_IMAGES,
    TRAIN_LABELS,
    IMAGE_SIZE
)

val_dataset = YOLODetectionDataset(
    VAL_IMAGES,
    VAL_LABELS,
    IMAGE_SIZE
)

print(
    "Training images   :",
    len(train_dataset)
)

print(
    "Validation images :",
    len(val_dataset)
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)


# ======================================================================
# 16. CREATE MODEL
# ======================================================================

print()
print("=" * 70)
print("CREATING ECOBOT-X TINY KD DETECTOR")
print("=" * 70)

model = EcoBotXTinyKDDetector(
    num_classes=NUM_CLASSES
).to(DEVICE)


# ======================================================================
# 17. LOAD SECTION 4 KD CHECKPOINT
# ======================================================================

print()
print("=" * 70)
print("LOADING SECTION 4 KD CHECKPOINT")
print("=" * 70)

print(
    "Checkpoint:",
    KD_CHECKPOINT
)

checkpoint = torch.load(
    KD_CHECKPOINT,
    map_location=DEVICE
)


# ------------------------------------------------------------
# Handle different checkpoint formats
# ------------------------------------------------------------

if isinstance(
    checkpoint,
    dict
):

    if "state_dict" in checkpoint:

        kd_state = checkpoint["state_dict"]

    elif "model_state_dict" in checkpoint:

        kd_state = checkpoint["model_state_dict"]

    elif "student_state_dict" in checkpoint:

        kd_state = checkpoint["student_state_dict"]

    else:

        kd_state = checkpoint

else:

    raise RuntimeError(
        "Unsupported KD checkpoint format."
    )


# ------------------------------------------------------------
# Remove possible DataParallel prefix
# ------------------------------------------------------------

clean_state = {}

for key, value in kd_state.items():

    new_key = key

    if new_key.startswith(
        "module."
    ):

        new_key = new_key[
            len("module.") :
        ]

    clean_state[new_key] = value


# ------------------------------------------------------------
# Load ONLY the student backbone/neck
# ------------------------------------------------------------

student_state = model.student.state_dict()

compatible_state = {}

skipped = []

for key, value in clean_state.items():

    if key in student_state:

        if (
            student_state[key].shape
            == value.shape
        ):

            compatible_state[key] = value

        else:

            skipped.append(
                (
                    key,
                    tuple(value.shape),
                    tuple(student_state[key].shape)
                )
            )


missing_before = [
    k for k in student_state
    if k not in compatible_state
]


model.student.load_state_dict(
    compatible_state,
    strict=False
)

print()
print(
    "[OK] KD weights loaded."
)

print(
    "Loaded tensors :",
    len(compatible_state)
)

print(
    "Missing tensors :",
    len(missing_before)
)

print(
    "Skipped shape mismatches :",
    len(skipped)
)

print()
print(
    "[IMPORTANT] Detection head is newly initialized."
)

print(
    "Backbone + Ghost-PAN initialized from Section 4 KD."
)


# ======================================================================
# 18. VERIFY FEATURE SHAPES
# ======================================================================

print()
print("=" * 70)
print("VERIFYING STUDENT FEATURES")
print("=" * 70)

model.eval()

dummy = torch.randn(
    1,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
    device=DEVICE
)

with torch.no_grad():

    features = model.student(
        dummy
    )

expected = {

    "P3": (
        1,
        64,
        80,
        80
    ),

    "P4": (
        1,
        96,
        40,
        40
    ),

    "P5": (
        1,
        128,
        20,
        20
    )
}

for name, shape in expected.items():

    actual = tuple(
        features[name].shape
    )

    print(
        f"{name}: {actual}"
    )

    if actual != shape:

        raise RuntimeError(
            f"{name} shape mismatch. "
            f"Expected {shape}, got {actual}"
        )

print()
print(
    "[OK] Exact Section 2 feature structure confirmed."
)


# ======================================================================
# 19. PARAMETER INFORMATION
# ======================================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

student_params = sum(
    p.numel()
    for p in model.student.parameters()
)

head_params = (
    total_params -
    student_params
)

print()
print("=" * 70)
print("MODEL INFORMATION")
print("=" * 70)

print(
    f"Student parameters : {student_params:,}"
)

print(
    f"Detection head     : {head_params:,}"
)

print(
    f"Total parameters    : {total_params:,}"
)

print(
    f"Total parameters M  : {total_params / 1e6:.3f}"
)

print(
    f"FP32 parameter size : "
    f"{total_params * 4 / (1024 ** 2):.3f} MB"
)


# ======================================================================
# 20. FREEZE / UNFREEZE STRATEGY
# ======================================================================
#
# The Section 4 KD weights already contain useful feature
# representations.
#
# We initially use a small learning rate for the student
# and a larger learning rate for the new detection head.
#
# ======================================================================

for param in model.student.parameters():

    param.requires_grad = True


for param in model.head_p3.parameters():

    param.requires_grad = True


for param in model.head_p4.parameters():

    param.requires_grad = True


for param in model.head_p5.parameters():

    param.requires_grad = True


# ======================================================================
# 21. OPTIMIZER
# ======================================================================

optimizer = torch.optim.AdamW(

    [

        {
            "params":
                model.student.parameters(),

            "lr":
                BACKBONE_LR
        },

        {
            "params":
                list(model.head_p3.parameters())
                +
                list(model.head_p4.parameters())
                +
                list(model.head_p5.parameters()),

            "lr":
                LEARNING_RATE
        }

    ],

    weight_decay=WEIGHT_DECAY
)


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=EPOCHS,

    eta_min=1e-6
)


# ======================================================================
# 22. BOX UTILITIES
# ======================================================================

def xywhn_to_xyxy(
    boxes
):

    xc = boxes[:, 0]
    yc = boxes[:, 1]
    w = boxes[:, 2]
    h = boxes[:, 3]

    x1 = xc - w / 2
    y1 = yc - h / 2

    x2 = xc + w / 2
    y2 = yc + h / 2

    return torch.stack(
        [
            x1,
            y1,
            x2,
            y2
        ],
        dim=1
    )


def box_iou(
    boxes1,
    boxes2
):

    if boxes1.numel() == 0:
        return torch.zeros(
            (0, boxes2.shape[0]),
            device=boxes1.device
        )

    if boxes2.numel() == 0:
        return torch.zeros(
            (boxes1.shape[0], 0),
            device=boxes1.device
        )

    area1 = (
        (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0)
        *
        (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    )

    area2 = (
        (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0)
        *
        (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)
    )

    lt = torch.maximum(
        boxes1[:, None, :2],
        boxes2[None, :, :2]
    )

    rb = torch.minimum(
        boxes1[:, None, 2:],
        boxes2[None, :, 2:]
    )

    wh = (
        rb - lt
    ).clamp(
        min=0
    )

    inter = (
        wh[:, :, 0]
        *
        wh[:, :, 1]
    )

    union = (
        area1[:, None]
        +
        area2[None, :]
        -
        inter
    )

    return inter / (
        union + 1e-7
    )


# ======================================================================
# 23. ANCHOR-FREE TARGET ASSIGNMENT
# ======================================================================

LEVELS = [
    ("P3", 8),
    ("P4", 16),
    ("P5", 32)
]


def choose_level(
    w,
    h
):

    size = max(
        w,
        h
    ) * IMAGE_SIZE

    if size <= 128:

        return 0

    elif size <= 256:

        return 1

    else:

        return 2


def build_targets(
    targets,
    level,
    device
):

    stride = LEVELS[level][1]

    grid = IMAGE_SIZE // stride

    cls_target = torch.zeros(
        (
            NUM_CLASSES,
            grid,
            grid
        ),
        device=device
    )

    box_target = torch.zeros(
        (
            4,
            grid,
            grid
        ),
        device=device
    )

    center_target = torch.zeros(
        (
            1,
            grid,
            grid
        ),
        device=device
    )

    positive = torch.zeros(
        (
            grid,
            grid
        ),
        dtype=torch.bool,
        device=device
    )

    if targets.numel() == 0:

        return (
            cls_target,
            box_target,
            center_target,
            positive
        )

    for target in targets:

        cls = int(
            target[0].item()
        )

        xc = target[1].item()
        yc = target[2].item()

        w = target[3].item()
        h = target[4].item()

        selected_level = choose_level(
            w,
            h
        )

        if selected_level != level:
            continue

        gx = xc * grid
        gy = yc * grid

        ix = int(
            min(
                max(
                    math.floor(gx),
                    0
                ),
                grid - 1
            )
        )

        iy = int(
            min(
                max(
                    math.floor(gy),
                    0
                ),
                grid - 1
            )
        )

        x1 = (
            xc -
            w / 2
        ) * IMAGE_SIZE

        y1 = (
            yc -
            h / 2
        ) * IMAGE_SIZE

        x2 = (
            xc +
            w / 2
        ) * IMAGE_SIZE

        y2 = (
            yc +
            h / 2
        ) * IMAGE_SIZE

        center_x = (
            (ix + 0.5)
            * stride
        )

        center_y = (
            (iy + 0.5)
            * stride
        )

        left = (
            center_x -
            x1
        ) / stride

        top = (
            center_y -
            y1
        ) / stride

        right = (
            x2 -
            center_x
        ) / stride

        bottom = (
            y2 -
            center_y
        ) / stride

        if min(
            left,
            top,
            right,
            bottom
        ) <= 0:

            continue

        cls_target[
            cls,
            iy,
            ix
        ] = 1.0

        box_target[
            :,
            iy,
            ix
        ] = torch.tensor(
            [
                left,
                top,
                right,
                bottom
            ],
            device=device
        )

        # FCOS-style centerness
        lr_min = min(
            left,
            right
        )

        lr_max = max(
            left,
            right
        )

        tb_min = min(
            top,
            bottom
        )

        tb_max = max(
            top,
            bottom
        )

        centerness = math.sqrt(

            (
                lr_min /
                (lr_max + 1e-7)
            )
            *
            (
                tb_min /
                (tb_max + 1e-7)
            )

        )

        center_target[
            0,
            iy,
            ix
        ] = centerness

        positive[
            iy,
            ix
        ] = True

    return (
        cls_target,
        box_target,
        center_target,
        positive
    )


# ======================================================================
# 24. DETECTION LOSS
# ======================================================================

def detection_loss(
    predictions,
    targets
):

    total_cls = torch.tensor(
        0.0,
        device=DEVICE
    )

    total_box = torch.tensor(
        0.0,
        device=DEVICE
    )

    total_center = torch.tensor(
        0.0,
        device=DEVICE
    )

    positive_count = 0

    for batch_index in range(
        len(targets)
    ):

        for level_index, (
            level_name,
            stride
        ) in enumerate(LEVELS):

            pred = predictions[
                level_name
            ]

            box_pred = pred["box"][
                batch_index
            ]

            cls_pred = pred["cls"][
                batch_index
            ]

            center_pred = pred["center"][
                batch_index
            ]

            (
                cls_target,
                box_target,
                center_target,
                positive
            ) = build_targets(

                targets[
                    batch_index
                ].to(DEVICE),

                level_index,

                DEVICE
            )

            # ----------------------------------------------------------
            # Classification
            # ----------------------------------------------------------

            cls_loss = F.binary_cross_entropy_with_logits(

                cls_pred,

                cls_target,

                reduction="mean"
            )

            total_cls += cls_loss

            # ----------------------------------------------------------
            # Centerness
            # ----------------------------------------------------------

            center_loss = F.binary_cross_entropy_with_logits(

                center_pred,

                center_target,

                reduction="mean"
            )

            total_center += center_loss

            # ----------------------------------------------------------
            # Box regression
            # ----------------------------------------------------------

            if positive.any():

                pred_box = box_pred[
                    :,
                    positive
                ].T

                true_box = box_target[
                    :,
                    positive
                ].T

                box_loss = F.smooth_l1_loss(

                    pred_box,

                    true_box,

                    reduction="mean"
                )

                total_box += box_loss

                positive_count += int(
                    positive.sum().item()
                )

    batch_size = len(targets)

    total_loss = (

        1.0 * total_cls / batch_size

        +

        2.0 * total_box / max(
            batch_size,
            1
        )

        +

        1.0 * total_center / batch_size

    )

    return (
        total_loss,
        total_cls.detach() / batch_size,
        total_box.detach() / max(
            batch_size,
            1
        ),
        total_center.detach() / batch_size,
        positive_count
    )


# ======================================================================
# 25. TRAINING FUNCTION
# ======================================================================

def train_one_epoch():

    model.train()

    running_loss = 0.0

    running_cls = 0.0

    running_box = 0.0

    running_center = 0.0

    total_positive = 0

    batches = 0

    start = time.time()

    for images, targets, paths in train_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        predictions = model(
            images
        )

        (
            loss,
            cls_loss,
            box_loss,
            center_loss,
            positive
        ) = detection_loss(
            predictions,
            targets
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=10.0
        )

        optimizer.step()

        running_loss += loss.item()

        running_cls += cls_loss.item()

        running_box += box_loss.item()

        running_center += center_loss.item()

        total_positive += positive

        batches += 1

    elapsed = time.time() - start

    return {

        "loss":
            running_loss / max(
                batches,
                1
            ),

        "cls_loss":
            running_cls / max(
                batches,
                1
            ),

        "box_loss":
            running_box / max(
                batches,
                1
            ),

        "center_loss":
            running_center / max(
                batches,
                1
            ),

        "positive":
            total_positive,

        "time":
            elapsed
    }


# ======================================================================
# 26. VALIDATION FUNCTION
# ======================================================================

@torch.no_grad()
def validate():

    model.eval()

    running_loss = 0.0

    batches = 0

    for images, targets, paths in val_loader:

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        predictions = model(
            images
        )

        (
            loss,
            cls_loss,
            box_loss,
            center_loss,
            positive
        ) = detection_loss(
            predictions,
            targets
        )

        running_loss += loss.item()

        batches += 1

    return (
        running_loss /
        max(
            batches,
            1
        )
    )


# ======================================================================
# 27. CHECKPOINT FUNCTION
# ======================================================================

def save_checkpoint(
    path,
    epoch,
    best_val_loss
):

    torch.save(

        {
            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "student_state_dict":
                model.student.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "best_val_loss":
                best_val_loss,

            "num_classes":
                NUM_CLASSES,

            "class_names":
                CLASS_NAMES,

            "image_size":
                IMAGE_SIZE,

            "architecture":
                "EcoBotX-Tiny-KD",

            "backbone":
                "GhostConv + DWConv + Coordinate Attention",

            "neck":
                "Lightweight Ghost-PAN",

            "head":
                "Anchor-Free Detection Head"

        },

        path
    )


# ======================================================================
# 28. TRAINING
# ======================================================================

print()
print("=" * 70)
print("STARTING SUPERVISED FINE-TUNING")
print("=" * 70)

print(
    "Epochs       :",
    EPOCHS
)

print(
    "Batch size   :",
    BATCH_SIZE
)

print(
    "Head LR      :",
    LEARNING_RATE
)

print(
    "Student LR   :",
    BACKBONE_LR
)

print(
    "Device       :",
    DEVICE
)

print()

best_val_loss = float("inf")

history = []

training_start = time.time()


for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = time.time()

    train_stats = train_one_epoch()

    val_loss = validate()

    scheduler.step()

    epoch_time = (
        time.time()
        -
        epoch_start
    )

    current_lr_head = (
        optimizer.param_groups[1]["lr"]
    )

    current_lr_student = (
        optimizer.param_groups[0]["lr"]
    )

    print(
        "-" * 70
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS}"
    )

    print(
        f"Train Loss      : "
        f"{train_stats['loss']:.6f}"
    )

    print(
        f"  Classification : "
        f"{train_stats['cls_loss']:.6f}"
    )

    print(
        f"  Box            : "
        f"{train_stats['box_loss']:.6f}"
    )

    print(
        f"  Centerness     : "
        f"{train_stats['center_loss']:.6f}"
    )

    print(
        f"Validation Loss : "
        f"{val_loss:.6f}"
    )

    print(
        f"Positive cells  : "
        f"{train_stats['positive']}"
    )

    print(
        f"Head LR         : "
        f"{current_lr_head:.8f}"
    )

    print(
        f"Student LR      : "
        f"{current_lr_student:.8f}"
    )

    print(
        f"Epoch Time      : "
        f"{epoch_time:.2f} sec"
    )

    if DEVICE.type == "cuda":

        allocated = (
            torch.cuda.memory_allocated()
            /
            1024**3
        )

        reserved = (
            torch.cuda.memory_reserved()
            /
            1024**3
        )

        print(
            f"GPU Memory      : "
            f"{allocated:.2f} GB allocated | "
            f"{reserved:.2f} GB reserved"
        )

    history.append(

        {
            "epoch":
                epoch,

            "train_loss":
                train_stats["loss"],

            "classification_loss":
                train_stats["cls_loss"],

            "box_loss":
                train_stats["box_loss"],

            "centerness_loss":
                train_stats["center_loss"],

            "validation_loss":
                val_loss,

            "learning_rate_head":
                current_lr_head,

            "learning_rate_student":
                current_lr_student,

            "epoch_time":
                epoch_time
        }
    )

    # --------------------------------------------------------------
    # Best model
    # --------------------------------------------------------------

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_path = (
            BEST_DIR /
            "best_student_detector.pt"
        )

        save_checkpoint(

            best_path,

            epoch,

            best_val_loss
        )

        print()

        print(
            "[BEST MODEL SAVED]"
        )

        print(
            best_path
        )

    # --------------------------------------------------------------
    # Periodic checkpoint
    # --------------------------------------------------------------

    if (
        epoch % SAVE_PERIOD == 0
        or
        epoch == EPOCHS
    ):

        checkpoint_path = (

            CHECKPOINT_DIR /
            f"student_detector_epoch_{epoch:03d}.pt"

        )

        save_checkpoint(

            checkpoint_path,

            epoch,

            best_val_loss
        )

        print()

        print(
            "[CHECKPOINT SAVED]"
        )

        print(
            checkpoint_path
        )


# ======================================================================
# 29. SAVE FINAL MODEL
# ======================================================================

total_training_time = (
    time.time()
    -
    training_start
)

final_path = (
    FINAL_DIR /
    "final_student_detector.pt"
)

save_checkpoint(

    final_path,

    EPOCHS,

    best_val_loss
)


history_path = (
    OUTPUT_ROOT /
    "training_history.json"
)

with open(
    history_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        history,
        f,
        indent=2
    )


# ======================================================================
# 30. FINAL SUMMARY
# ======================================================================

print()
print("=" * 70)
print("SECTION 5 — SUPERVISED FINE-TUNING COMPLETED")
print("=" * 70)

print()

print(
    "Epochs completed :",
    EPOCHS
)

print(
    "Best validation loss :",
    f"{best_val_loss:.6f}"
)

print(
    "Total training time :",
    f"{total_training_time / 3600:.2f} hours"
)

print()

print(
    "Best detector :",
    best_path
)

print(
    "Final detector :",
    final_path
)

print(
    "Training history :",
    history_path
)

print()
print("=" * 70)
print("[OK] SECTION 5 COMPLETED SUCCESSFULLY")
print("=" * 70)

ECOBOT-X KNOWLEDGE DISTILLATION
SECTION 5 — DETECTION HEAD + SUPERVISED FINE-TUNING

Device : cuda:0
GPU    : NVIDIA GeForce RTX 3050 Laptop GPU
CUDA   : 12.8

PATH VALIDATION
[OK] Train images: G:\EcoBotX_YOLO\images\train
[OK] Train labels: G:\EcoBotX_YOLO\labels\train
[OK] Validation images: G:\EcoBotX_YOLO\images\val
[OK] Validation labels: G:\EcoBotX_YOLO\labels\val
[OK] Test images: G:\EcoBotX_YOLO\images\test
[OK] Test labels: G:\EcoBotX_YOLO\labels\test
[OK] KD checkpoint: G:\EcoBotX_YOLO_training\knowledge_distillation\section4_feature_kd\best_student_kd.pt

LOADING DATASETS
Training images   : 3563
Validation images : 891

CREATING ECOBOT-X TINY KD DETECTOR

LOADING SECTION 4 KD CHECKPOINT
Checkpoint: G:\EcoBotX_YOLO_training\knowledge_distillation\section4_feature_kd\best_student_kd.pt

[OK] KD weights loaded.
Loaded tensors : 340
Missing tensors : 0
Skipped shape mismatches : 0

[IMPORTANT] Detection head is newly initialized.
Backbone + Ghost-PAN initialized from Section 4